# 第 80 课：推理服务发布门禁、Runbook 与事故复盘自动化

前一课已经能从真实增量 KV 推理的 SLO 证据触发回滚。本课补上**发布之前、发布之中、回滚之后**的运行机制：

**权重身份与资源预检 → 灰度阶段可复查证据 → 双职责审批 → 原子活动指针 → 故障后对账 → 新流量验证 → 无责复盘与行动项。**

核心问题不是“怎么把一个开关从 stable 改成 canary”，而是**凭什么允许这次改变、改变一半崩溃后怎么知道实际状态、怎样区分已执行动作与尚未确认的根因**。

课程完全离线、CPU、PyTorch，包含真实 KV 张量的增量 attention，不调用云平台。版本身份、审批与发布指针均为教学模拟，绝不更改真实服务。

## 0. 运行方式与验收边界

从全新内核顺序运行。默认跑完整七个演练，创建新的时间戳目录；可先将 `RUN_FULL_EXPERIMENT=False` 只学习预检、门禁、审批与练习。

同名独立脚本：

```bash
python lesson_80_release_gates_runbook_postmortem.py --checks-only
python lesson_80_release_gates_runbook_postmortem.py --math-only
python lesson_80_release_gates_runbook_postmortem.py
python lesson_80_release_gates_runbook_postmortem.py --reload-run lesson_80_outputs/实际运行目录
```

需要已有 PyTorch 环境，不下载模型、连接外部账户或服务。CPU 处理的实际 KV attention 是真的，但**延迟是确定性的逻辑 tick，不是 GPU 毫秒，也没有真实 HTTP、并发分布式发布与网络攻击防护**。

本课提供完整实现、158 项回归检查、三道默认关闭的独立练习与完整参考答案。正常运行检查通过不等于真实生产系统安全认证。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第80课：推理服务发布门禁、Runbook、事故复盘与事务恢复。

PyTorch 小型随机 Transformer，真实 CPU 增量 KV 推理、确定性逻辑 tick。
默认完全离线，无真实 HTTP/公网服务/生产发布权限，不自动下载模型。
演示独立角色审批、证据新鲜度、原子指针、故障后对账及无责复盘。
这里的审核角色是*教学模拟*，不是身份认证；哈希链也不是数字签名。
"""
from __future__ import annotations

# %% 01 环境、安全边界与确定性证据
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('本课需要已有PyTorch环境；不会自动安装、下载或发布。') from exc

VERSION='lesson80-release-gate-v1'
SOURCES={
    'sre_canary':'https://sre.google/workbook/canarying-releases/',
    'sre_postmortem':'https://sre.google/sre-book/postmortem-culture/',
    'sre_response':'https://sre.google/resources/practices-and-processes/incident-management-guide/',
    'aws_rollback':'https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html',
    'pytorch_reproducibility':'https://docs.pytorch.org/docs/stable/notes/randomness.html',
}
class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(8001)
RUN_FULL_EXPERIMENT = True
OUTPUT_ROOT = Path('lesson_80_outputs')
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__,'CUDA可用:',torch.cuda.is_available())
print('默认执行：离线CPU、逻辑tick、无真实发布权限。')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
默认执行：离线CPU、逻辑tick、无真实发布权限。


## 1. 发布门禁不是“审核通过”的字符串

我们必须把证据和**同一个计划**绑定：发布ID、稳定模型摘要、候选 adapter 对应的完整模型摘要、tokenizer/Chat Template、灰度 epoch、门禁策略版本。任何一个变化后都不能继续复用旧证据。

本课 `ReleaseIntent` 是发布请求；`GatePolicy` 是提前确定的决策阈值。两者均作为哈希输入。**摘要仅能检查意外内容变化，不是签名。** 一个能够重写所有文件的攻击者也能重算摘要；本课没有外部可信证据签发方。

我们不在这一步自动将候选升级为“稳定模型”，先执行真正的模型与服务预检。

In [3]:
# %% 02 离线小型Transformer、LoRA与教学分词器
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

## 2. 不用随机指标伪造成功：真实 KV 张量能否对齐全上下文？

本课沿用小型因果 Transformer、Q/V LoRA 和离线教学 tokenizer。增量缓存 backing Tensor 有**实际 K/V 数值**，形状为 `[block, layer, K_or_V, slot, head, head_dim]`。每步计算新 Q/K/V、写入 KV，再对历史 KV 做 attention。

完整前向与增量前向必须得到相同的末位置 logits（合理浮点容差内）。不同 adapter 版本不能共享同一个会话的 KV；取消或完成后物理页引用数与零化状态也要对齐。

这不是 vLLM 内核，prefill 仍逐 token 执行，没有测算吞吐或 GPU 显存。

In [4]:
# %% 03 真实物理KV张量、增量解码与资源账本
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None

class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""

class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

## 3. 真正的请求准入、灰度路由与逻辑 tick

版本在**准入时固定**，同一租户同一幂等键的相同请求不会重复执行，不会额外增加唯一终态分母。不同参数复用同一键会明确报错。Trace 只有短摘要和低基数结构字段，不写请求明文到门禁证据。

网关模拟 `slow`（候选专属逻辑开销）、`global_slow`（稳定和候选都慢）与 `kv_leak`（候选有物理页未释放）。每条响应仍由真实模型执行。`duration_ticks` 是教学逻辑工作量，不能写成 HTTP latency 或 GPU milliseconds。

SRE 的灰度实践强调用一小部分流量对变更做受控评估，而不是一次性切换所有请求。[Google SRE：Canarying Releases](https://sre.google/workbook/canarying-releases/)

In [5]:
# %% 04 幂等准入、固定版本路由与模拟流量
@dataclass(frozen=True)
class ServiceRequest:
    tenant: str
    key: str
    text: str = 'hello'
    timeout_ticks: int = 70
    max_new_tokens: int = 3
    template: str = 'chat-v1'

    def validate(self):
        for name in ('tenant','key','template'):
            v=getattr(self,name)
            require(type(v) is str and bool(re.fullmatch(r'[A-Za-z0-9_-]{1,48}',v)),name+'格式非法')
        bounded_str(self.text,128,'text')
        require(bool(self.text),'请求正文不能为空')
        require(type(self.timeout_ticks) is int and 1<=self.timeout_ticks<=1000,'timeout_ticks非法')
        require(type(self.max_new_tokens) is int and 1<=self.max_new_tokens<=6,'max_new_tokens非法')

    def fingerprint(self):
        self.validate()
        return digest(asdict(self))

@dataclass
class Admission:
    version: str
    epoch: int
    fingerprint: str
    trace_id: str
    request: ServiceRequest
    terminal: dict | None = None
    completed: bool = False

class CanaryRouter:
    def __init__(self,stable='stable',canary='canary',weight=0):
        require(stable!=canary and all(type(v) is str and v for v in (stable,canary)),'版本非法')
        self.stable=stable;self.canary=canary
        self.weight=0;self.epoch=0;self.latched=False
        self.set_weight(weight)

    @staticmethod
    def bucket(tenant,key):
        return int(hashlib.sha256((tenant+'|'+key).encode('utf-8')).hexdigest()[:8],16)%100

    def choose(self,req:ServiceRequest):
        req.validate()
        return self.canary if self.bucket(req.tenant,req.key)<self.weight else self.stable

    def set_weight(self,weight):
        require(type(weight) is int and 0<=weight<=100,'灰度权重必须为0..100整数')
        require(not self.latched or weight==0,'回滚锁存后不得自动开启灰度')
        self.weight=weight

    def start_canary(self,weight):
        require(self.weight==0 and not self.latched,'已有灰度或处于回滚锁存状态')
        self.set_weight(weight)
        require(self.weight>0,'启用灰度权重必须为正')
        self.epoch+=1
        return self.epoch

    def rollback(self):
        if self.latched:return False
        require(self.weight>0,'没有灰度版本可以回滚')
        self.weight=0;self.latched=True;self.epoch+=1
        return True

    def manual_reenable(self,approved:bool,postcheck:bool,weight:int):
        require(approved is True and postcheck is True,'重新开启灰度需要人工批准和恢复验证')
        require(self.latched,'当前没有回滚锁存状态')
        require(type(weight) is int and 1<=weight<=100,'新灰度权重非法')
        self.latched=False;self.weight=weight;self.epoch+=1

class SafeAudit:
    """不记录正文或主观归因；哈希链检测偶发篡改，不是签名或不可抵赖机制。"""
    def __init__(self):
        self.rows=[]

    def append(self,event:str,details:dict):
        require(event in {'canary_start','alert','rollback','operator_page','postcheck','manual_reenable','release_leaks'},
                '审计事件类型非法')
        require(type(details) is dict and len(canonical(details))<=3000,'审计细节不合法')
        previous=self.rows[-1]['hash'] if self.rows else '0'*64
        row={'seq':len(self.rows),'event':event,'details':copy.deepcopy(details),'prev':previous}
        row['hash']=digest(row)
        self.rows.append(row)
        return row['hash']

    def verify(self):
        for i,row in enumerate(self.rows):
            require(row['seq']==i,'审计序号错位')
            require(row['prev']==(self.rows[i-1]['hash'] if i else '0'*64),'审计哈希链断裂')
            require(row['hash']==digest({k:v for k,v in row.items() if k!='hash'}),'审计记录被改写')
        return {'count':len(self.rows),'root_hash':self.rows[-1]['hash'] if self.rows else '0'*64}

def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

@dataclass(frozen=True)
class DrillConfig:
    capacity_blocks: int=32
    block_size: int=4
    canary_weight: int=50
    slo_target: float=.95
    latency_target_ticks: int=25
    short_n: int=6
    long_n: int=12
    min_excess_bad: float=.25
    trigger_burn: float=4.0
    max_stable_bad: float=.20
    leak_pressure_fraction: float=.35
    leak_min_sessions: int=3
    max_version_staleness: int=12
    recover_samples: int=12

    def validate(self):
        ints=('capacity_blocks','block_size','latency_target_ticks','short_n','long_n',
              'leak_min_sessions','max_version_staleness','recover_samples')
        require(all(type(getattr(self,k)) is int and getattr(self,k)>0 for k in ints),'整数参数必须为正')
        require(type(self.canary_weight) is int and 1<=self.canary_weight<=99,'灰度权重限制1..99')
        require(self.long_n>=self.short_n>=2,'窗口长度顺序错误')
        for k in ('slo_target','min_excess_bad','max_stable_bad','leak_pressure_fraction'):
            v=getattr(self,k)
            require(type(v) in (float,int) and math.isfinite(v) and 0<v<1,k+'必须在0..1')
        require(type(self.trigger_burn) in (float,int) and math.isfinite(self.trigger_burn)
                and self.trigger_burn>0,'burn阈值错误')

class IncrementalGateway:
    """只有准入/逻辑时间模拟，KV与attention是真实CPU tensor。无HTTP、无网络和异步并发。"""
    def __init__(self,tok,models,config:DrillConfig):
        config.validate()
        require(set(models)=={'stable','canary'},'需要stable/canary两个版本')
        self.tok=tok;self.models=models;self.config=config
        self.router=CanaryRouter(weight=0)
        self.pool=PagedKVPool(config.capacity_blocks,config.block_size,
                              models['stable'].cfg,set(models))
        self.runner=CachedModelRunner(tok,models,self.pool)
        self.registry={}
        self.terminal=[]
        self.traces=[]
        self.logical_tick=0
        self.execution_count=0
        self.leaked_sessions={}
        self.baseline_hash=state_hash(models['stable'].state_dict())
        self.canary_hash=state_hash(models['canary'].state_dict())
        self.peak_live_blocks=0
        self.audit=SafeAudit()

    def admit(self,req:ServiceRequest):
        req.validate();identity=(req.tenant,req.key)
        fp=req.fingerprint()
        if identity in self.registry:
            old=self.registry[identity]
            require(old.fingerprint==fp,'同租户幂等键冲突：不得重新执行')
            return old,False
        version=self.router.choose(req)
        trace=digest(['trace',req.tenant,req.key])[:32]
        rec=Admission(version,self.router.epoch,fp,trace,req)
        self.registry[identity]=rec
        return rec,True

    def _safe_terminal(self,rec,status,latency,generated,kv_peak,cache_diff,stage):
        require(not rec.completed,'同一请求不允许重复终态')
        good=(status=='ok' and latency<=self.config.latency_target_ticks)
        row={'ordinal':len(self.terminal),'trace_id':rec.trace_id,
             'version':rec.version,'admitted_epoch':rec.epoch,
             'terminal_status':status,'duration_ticks':latency,'slo_good':good,
             'generated_tokens':generated,'kv_peak_blocks':kv_peak,
             'cache_full_max_difference':cache_diff,'stage':stage}
        self.terminal.append(row)
        rec.terminal=row;rec.completed=True
        # Trace ID仅作诊断连接键，不用于指标标签。原始文本/tenant/key不落日志。
        self.traces.append({'trace_id':rec.trace_id,'span':'inference','version':rec.version,
                            'stage':stage,'status':status,'duration_ticks':latency,
                            'prompt_digest':digest(rec.request.text)[:16]})
        return row

    def execute(self,rec:Admission,fault='none'):
        require(self.registry.get((rec.request.tenant,rec.request.key)) is rec,
                '请求不是当前服务正式准入的对象')
        if rec.completed:return rec.terminal,False
        require(fault in {'none','slow','internal','kv_leak','global_slow'},'非法故障模式')
        require(state_hash(self.models['stable'].state_dict())==self.baseline_hash,'稳定版本被改写')
        require(state_hash(self.models['canary'].state_dict())==self.canary_hash,'灰度版本被改写')
        self.execution_count+=1
        sid='req'+str(self.execution_count)
        prompt_ids=self.tok.apply_chat_template([{'role':'user','content':rec.request.text}],
                                                 tokenize=True,add_generation_prompt=True)
        require(len(prompt_ids)+rec.request.max_new_tokens<=self.models[rec.version].cfg.max_length,
                '上下文越界不静默截断')
        slow=(fault=='global_slow' or (fault=='slow' and rec.version=='canary'))
        deliberately_leak=(fault=='kv_leak' and rec.version=='canary')
        internal=(fault=='internal' and rec.version=='canary')
        latency=math.ceil(len(prompt_ids)/3)
        generated=0;max_diff=0.0
        status='ok';stage='prefill';opened=False;leaked=False
        try:
            self.runner.open_prefill(sid,rec.version,prompt_ids)
            opened=True
            max_diff=max(max_diff,self.runner.assert_consistent(sid))
            if internal:raise RuntimeError('injected_internal_failure')
            stage='decode'
            for _ in range(rec.request.max_new_tokens):
                logits=self.runner.last_logits[sid]
                # 屏蔽无效的结构token，避免把控制ID当作正文去解码。
                scores=logits.clone()
                scores[:16]=-torch.inf
                next_id=int(scores.argmax())
                self.runner.step(sid,next_id)
                generated+=1
                latency+=2
                if next_id==self.tok.eos_token_id:break
            max_diff=max(max_diff,self.runner.assert_consistent(sid))
            if slow:latency+=40  # 故意注入逻辑开销；不sleep、不声称真实RTT
            if latency>rec.request.timeout_ticks:
                status='deadline_exceeded'
            if deliberately_leak:
                self.leaked_sessions[sid]='canary'
                leaked=True
        except NoKVSpace:
            status='capacity_rejected';stage='kv_allocation'
        except RuntimeError as exc:
            require(str(exc)=='injected_internal_failure','未知运行错误必须向上抛出')
            status='internal_error';stage='model_runtime'
        finally:
            if sid in self.pool.sessions and not leaked:
                s=self.pool.sessions[sid]
                if s.pending is not None:self.pool.abort(sid)
                self.runner.close(sid)
            self.pool.audit()
        self.peak_live_blocks=max(self.peak_live_blocks,self.pool.usage()['live_blocks'])
        # 模拟中每个请求推进一个到达顺序tick，latency是单请求的独立逻辑工作量。
        self.logical_tick+=1
        return self._safe_terminal(rec,status,latency,generated,
                                   self.peak_live_blocks,max_diff,stage),True

    def serve(self,req:ServiceRequest,fault='none'):
        rec,new=self.admit(req)
        if not new and rec.completed:return rec.terminal,False
        return self.execute(rec,fault)

    def release_leaks(self):
        items=list(self.leaked_sessions)
        for sid in items:
            require(sid in self.pool.sessions,'泄漏记录找不到真实会话')
            self.runner.close(sid)
            del self.leaked_sessions[sid]
        self.pool.audit()
        return len(items)

    def metrics(self):
        counter=Counter((r['version'],r['terminal_status']) for r in self.terminal)
        # 低基数维度仅保留服务版本及枚举状态，不包含tenant、key、trace_id或文本。
        return {'terminal_counts':[{'version':v,'status':s,'count':n}
                                   for (v,s),n in sorted(counter.items())],
                'attempts':len(self.registry),'real_model_executions':self.execution_count,
                'kv_live_blocks':self.pool.usage()['live_blocks'],
                'kv_leaked_sessions':len(self.leaked_sessions),
                'kv_peak_live_blocks':self.peak_live_blocks,
                'kv_bytes_per_block':self.pool.bytes_per_block}

def window_sli(rows:list[dict],n:int,target:float):
    require(type(n) is int and n>0 and 0<target<1,'窗口/目标非法')
    chosen=rows[-n:]
    if len(chosen)<n:return {'available':False,'count':len(chosen),'required':n}
    bad=sum(not r['slo_good'] for r in chosen)
    return {'available':True,'count':n,'bad':bad,'good':n-bad,
            'bad_rate':bad/n,'burn':(bad/n)/(1-target)}

def find_keys(router:CanaryRouter,version:str,n:int,used:set[str],start=0,tenant='orgA'):
    require(version in {'stable','canary'} and type(n) is int and n>0,'目标版本/数量错误')
    keys=[];i=start
    while len(keys)<n:
        key=f'k{i:05d}';i+=1
        if key in used:continue
        if router.choose(ServiceRequest(tenant,key))==version:
            used.add(key);keys.append(key)
    return keys

def stage_traffic(gateway:IncrementalGateway,counts:dict,used:set,mode='none',tenant='orgA'):
    """控制组流量数量：用于可解释的配对故障演练，不代表真实在线流量分布。"""
    require(set(counts)<={'stable','canary'},'未知流量组')
    keys=[]
    for version,count in counts.items():
        if count:keys.extend((version,k) for k in find_keys(gateway.router,version,count,used,tenant=tenant))
    # 先交错排列，防止稳定版本全部排在最后后才进行观察。
    stable=[k for v,k in keys if v=='stable'];canary=[k for v,k in keys if v=='canary']
    ordered=[]
    for i in range(max(len(stable),len(canary))):
        if i<len(stable):ordered.append(('stable',stable[i]))
        if i<len(canary):ordered.append(('canary',canary[i]))
    rows=[]
    for expected_version,key in ordered:
        row,_=gateway.serve(ServiceRequest(tenant,key,text='hello',timeout_ticks=70),fault=mode)
        require(row['version']==expected_version,'灰度路由结果与键分桶不一致')
        rows.append(row)
    return rows

def save_models(folder,tok,models):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    stable=models['stable']
    cfg=asdict(stable.cfg)
    foundation={k:t.cpu().detach().clone() for k,t in base_state(stable).items()}
    torch.save({'version':VERSION,'kind':'base','config':cfg,'tokenizer':tok.contract(),
                'state':foundation,'base_hash':state_hash(foundation)},folder/'base.pt')
    for key,model in models.items():
        torch.save({'version':VERSION,'kind':'adapter','name':key,'config':cfg,
                    'base_hash':state_hash(foundation),'tokenizer':tok.contract(),
                    'rank':4,'alpha':8.0,'state':adapter_snapshot(model),
                    'model_hash':state_hash(model.state_dict())},folder/(key+'.pt'))

def load_models(folder):
    folder=Path(folder)
    base=torch.load(folder/'base.pt',map_location='cpu',weights_only=True)
    require(base['version']==VERSION and base['kind']=='base','基座版本/类型错误')
    tok=ByteMergeChatTokenizer()
    require(base['tokenizer']==tok.contract(),'词表身份不匹配')
    require(state_hash(base['state'])==base['base_hash'],'基座内容摘要不匹配')
    reference=TinyChatLM(ToyConfig(**base['config']))
    reference.load_state_dict(base['state'],strict=True)
    reference.eval().requires_grad_(False)
    models={}
    for name in ('stable','canary'):
        value=torch.load(folder/(name+'.pt'),map_location='cpu',weights_only=True)
        require(value['version']==VERSION and value['kind']=='adapter' and value['name']==name,'adapter类型/版本错误')
        require(value['config']==base['config'] and value['base_hash']==base['base_hash']
                and value['tokenizer']==tok.contract(),'adapter不是对应基座/词表')
        policy=inject_lora(reference,value['rank'],value['alpha'])
        restore_adapter(policy,value['state'])
        policy.eval().requires_grad_(False)
        require(state_hash(policy.state_dict())==value['model_hash'],'adapter权重内容摘要错误')
        models[name]=policy
    return tok,models

In [6]:
tok, models = build_servable_models(8001)
print('词表大小:',tok.vocab_size)
print('两个模型base相同:',base_hash(models['stable'])==base_hash(models['canary']))
print('模型权重身份不同:',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
print('两个版本全部冻结:',all(not p.requires_grad for m in models.values() for p in m.parameters()))

词表大小: 281
两个模型base相同: True
模型权重身份不同: True
两个版本全部冻结: True


In [7]:
probe_gateway=IncrementalGateway(tok,models,DrillConfig())
probe_gateway.router.start_canary(50)
probe_rows=stage_traffic(probe_gateway,{'stable':3,'canary':3},set(),'none')
print('六个真实执行的版本/状态/逻辑耗时：')
for r in probe_rows:
    print(r['version'],r['terminal_status'],r['duration_ticks'],r['cache_full_max_difference'])
print('实际KV物理占用:',probe_gateway.pool.usage()['live_blocks'])

六个真实执行的版本/状态/逻辑耗时：
stable ok 9 2.384185791015625e-07
canary ok 9 2.384185791015625e-07
stable ok 9 2.384185791015625e-07
canary ok 9 2.384185791015625e-07
stable ok 9 2.384185791015625e-07
canary ok 9 2.384185791015625e-07
实际KV物理占用: 0


## 4. 发布证据：保持两侧独立分母，且每个数有来源

正式门禁先要求每版本至少12条**唯一请求终态**，并且证据来自期望灰度 epoch；不能把旧暖机请求、客户端重试、不同候选权重或不同词表的报告拼在一起。

本课收集：每版本 `n/bad/errors/latency_exceeded`、模型和分词身份、KV 实际残留/峰值、增量/full 最大差异以及采样逻辑 tick。

按事先固定的 `GatePolicy` 判定：

- `allow`：两侧有足够且新鲜的样本、质量达标、KV与模型身份可信。
- `block`：候选退化、物理KV泄漏、KV对齐失败或峰值超预算。
- `page`：稳定对照也差，不能把共享问题当作只需回滚候选。
- `wait`：样本不足、窗口陈旧或 epoch 已变化。

这里是示例的固定请求数门禁，并非生产统计显著性检验。质量指标不是语义回答正确率；只检验模拟服务终态/SLO。

In [8]:
# %% 05 发布意图、可信边界与可重复的发布证据
@dataclass(frozen=True)
class GatePolicy:
    minimum_per_version: int = 12
    slo_target: float = .95
    maximum_bad_fraction: float = 0.10
    maximum_candidate_excess_bad: float = 0.10
    evidence_ttl_ticks: int = 30
    approval_ttl_ticks: int = 30
    require_kv_zero: bool = True
    cache_tolerance: float = 2e-5
    max_kv_peak_blocks: int = 32

    def validate(self):
        require(type(self.minimum_per_version) is int and self.minimum_per_version>=6,
                '每版本最小样本量至少6')
        require(type(self.evidence_ttl_ticks) is int and self.evidence_ttl_ticks>0,
                '证据TTL需要正整数tick')
        require(type(self.approval_ttl_ticks) is int and self.approval_ttl_ticks>0,
                '审批TTL需要正整数tick')
        for key in ('slo_target','maximum_bad_fraction','maximum_candidate_excess_bad'):
            x=getattr(self,key)
            require(type(x) in (int,float) and math.isfinite(x) and 0<x<1,key+'非法')
        require(type(self.cache_tolerance) in (int,float) and math.isfinite(self.cache_tolerance)
                and self.cache_tolerance>0,'KV一致性阈值非法')
        require(type(self.max_kv_peak_blocks) is int and self.max_kv_peak_blocks>0,
                'KV峰值预算非法')
        require(type(self.require_kv_zero) is bool,'是否零泄漏需bool')


@dataclass(frozen=True)
class ReleaseIntent:
    release_id: str
    stable_hash: str
    candidate_hash: str
    tokenizer_hash: str
    template_id: str
    author: str
    start_generation: int = 0
    expected_canary_epoch: int = 1

    def validate(self):
        for key in ('release_id','template_id','author'):
            value=getattr(self,key)
            require(type(value) is str and re.fullmatch(r'[A-Za-z0-9_-]{3,48}',value),
                    key+'格式非法')
        for key in ('stable_hash','candidate_hash','tokenizer_hash'):
            value=getattr(self,key)
            require(type(value) is str and re.fullmatch('[0-9a-f]{64}',value),key+'摘要非法')
        require(self.stable_hash!=self.candidate_hash,'候选不能与稳定版完全相同')
        require(type(self.start_generation) is int and self.start_generation>=0,'起始generation非法')
        require(type(self.expected_canary_epoch) is int and self.expected_canary_epoch>0,
                '灰度epoch非法')


def make_intent(tok,models,release_id='rel-80-demo',author='release-author'):
    return ReleaseIntent(release_id=release_id,stable_hash=state_hash(models['stable'].state_dict()),
                         candidate_hash=state_hash(models['canary'].state_dict()),
                         tokenizer_hash=digest(tok.contract()),template_id='chat-v1',author=author)


def capture_release_evidence(gateway:IncrementalGateway,intent:ReleaseIntent,
                             policy:GatePolicy) -> dict:
    """只捕获真实网关的唯一终态与物理资源；不保存请求、tenant、正文。"""
    policy.validate();intent.validate()
    require(gateway.router.epoch==intent.expected_canary_epoch and 0<gateway.router.weight<100,
            '发布证据必须来自预期epoch且两个版本均承载流量')
    require(state_hash(gateway.models['stable'].state_dict())==intent.stable_hash,
            '稳定权重不是发布计划绑定的权重')
    require(state_hash(gateway.models['canary'].state_dict())==intent.candidate_hash,
            '候选权重不是发布计划绑定的权重')
    require(digest(gateway.tok.contract())==intent.tokenizer_hash,'tokenizer已发生变化')
    rows=gateway.terminal
    require(len({r['trace_id'] for r in rows})==len(rows),'重复终态不能重复计入分母')
    require(all(r['admitted_epoch']==intent.expected_canary_epoch for r in rows),
            '不能把旧epoch的warmup终态混到发布证据')
    require(all(r['version'] in {'stable','canary'} for r in rows),'未知版本终态')
    counts={}
    for version in ('stable','canary'):
        relevant=[r for r in rows if r['version']==version]
        counts[version]={
            'n':len(relevant),
            'bad':sum(not r['slo_good'] for r in relevant),
            'latency_exceeded':sum(r['terminal_status']=='ok' and not r['slo_good'] for r in relevant),
            'errors':sum(r['terminal_status']!='ok' for r in relevant),
            'latest_ordinal':max((r['ordinal'] for r in relevant),default=-1),
        }
    kv=gateway.pool.usage()
    payload={
        'release_id':intent.release_id,
        'intent_hash':digest(asdict(intent)),
        'policy_hash':digest(asdict(policy)),
        'sample_epoch':gateway.router.epoch,
        'sample_tick':gateway.logical_tick,
        'terminal_count':len(rows),
        'unique_trace_count':len({r['trace_id'] for r in rows}),
        'last_ordinal':len(rows)-1,
        'by_version':counts,
        'stable_hash':state_hash(gateway.models['stable'].state_dict()),
        'candidate_hash':state_hash(gateway.models['canary'].state_dict()),
        'tokenizer_hash':digest(gateway.tok.contract()),
        'kv':{'live_blocks':kv['live_blocks'],'leaked_sessions':len(gateway.leaked_sessions),
              'peak_live_blocks':kv['peak_live_blocks'],
              'page_bytes':kv['bytes_per_block'],
              'capacity_blocks':kv['capacity_blocks']},
        'max_cache_full_diff':max((r['cache_full_max_difference'] for r in rows),default=None),
        'statuses':{v:dict(Counter(r['terminal_status'] for r in rows if r['version']==v))
                    for v in ('stable','canary')},
        'source':'deterministic CPU incremental-KV inference; synthetic logical ticks',
    }
    return {'payload':payload,'sha256':digest(payload)}


def gate_decision(evidence:dict,intent:ReleaseIntent,policy:GatePolicy,
                  *,current_tick:int,current_epoch:int,current_hashes:dict) -> dict:
    """策略完全固定，所有判定必须绑定当前权重/epoch，不接受裸数字或未绑定报告。"""
    policy.validate();intent.validate()
    require(type(evidence) is dict and set(evidence)=={'payload','sha256'},'证据形状错误')
    e=evidence['payload']
    require(type(e) is dict and evidence['sha256']==digest(e),'证据摘要不匹配')
    require(e['intent_hash']==digest(asdict(intent)) and e['policy_hash']==digest(asdict(policy)),
            '证据与意图或门禁策略不匹配')
    require(e['release_id']==intent.release_id and e['stable_hash']==intent.stable_hash
            and e['candidate_hash']==intent.candidate_hash and e['tokenizer_hash']==intent.tokenizer_hash,
            '证据模型/tokenizer身份不匹配')
    require(current_hashes=={'stable':intent.stable_hash,'canary':intent.candidate_hash},
            '发布时模型内容已漂移')
    require(type(current_tick) is int and current_tick>=e['sample_tick'],
            '当前时间不能早于采样时间')
    require(type(current_epoch) is int and current_epoch>=e['sample_epoch'],
            'epoch不能倒退')
    if (current_tick-e['sample_tick']>policy.evidence_ttl_ticks
            or current_epoch!=e['sample_epoch']
            or current_epoch!=intent.expected_canary_epoch):
        return {'action':'wait','reason':'stale_evidence_or_epoch','evidence_hash':evidence['sha256']}
    require(e['unique_trace_count']==e['terminal_count']
            and e['terminal_count']==sum(x['n'] for x in e['by_version'].values()),
            '唯一终态分母不守恒')
    stats=e['by_version']
    require(set(stats)=={'stable','canary'},'需要稳定与候选两侧')
    if any(stats[v]['n']<policy.minimum_per_version for v in stats):
        return {'action':'wait','reason':'insufficient_per_version_sample','evidence_hash':evidence['sha256']}
    latest=max(e['last_ordinal'],0)
    if any(latest-stats[v]['latest_ordinal']>policy.evidence_ttl_ticks for v in stats):
        return {'action':'wait','reason':'stale_per_version_telemetry','evidence_hash':evidence['sha256']}
    kv=e['kv']
    require(kv['capacity_blocks']>0 and kv['peak_live_blocks']>=kv['live_blocks']>=0
            and kv['peak_live_blocks']<=kv['capacity_blocks'], 'KV报告不守恒')
    if policy.require_kv_zero and (kv['live_blocks']>0 or kv['leaked_sessions']>0):
        return {'action':'block','reason':'unreleased_physical_kv','evidence_hash':evidence['sha256']}
    if kv['peak_live_blocks']>policy.max_kv_peak_blocks:
        return {'action':'block','reason':'peak_kv_exceeds_budget','evidence_hash':evidence['sha256']}
    if e['max_cache_full_diff'] is None or e['max_cache_full_diff']>policy.cache_tolerance:
        return {'action':'block','reason':'cache_vs_full_mismatch','evidence_hash':evidence['sha256']}
    bad_stable=stats['stable']['bad']/stats['stable']['n']
    bad_candidate=stats['canary']['bad']/stats['canary']['n']
    if bad_stable>policy.maximum_bad_fraction:
        return {'action':'page','reason':'control_also_unhealthy','evidence_hash':evidence['sha256']}
    if bad_candidate>policy.maximum_bad_fraction:
        return {'action':'block','reason':'candidate_slo_regression','evidence_hash':evidence['sha256']}
    if bad_candidate-bad_stable>policy.maximum_candidate_excess_bad:
        return {'action':'block','reason':'candidate_excess_bad','evidence_hash':evidence['sha256']}
    return {'action':'allow','reason':'fresh_healthy_bound_evidence','evidence_hash':evidence['sha256']}

In [9]:
policy = GatePolicy()
intent = make_intent(tok,models)
healthy_gateway = IncrementalGateway(tok,models,DrillConfig())
healthy_gateway.router.start_canary(50)
staged = stage_traffic(healthy_gateway,{'stable':12,'canary':12},set(),'none')
evidence = capture_release_evidence(healthy_gateway,intent,policy)
verdict = gate_decision(evidence,intent,policy,
                        current_tick=healthy_gateway.logical_tick,
                        current_epoch=healthy_gateway.router.epoch,
                        current_hashes={k:state_hash(m.state_dict()) for k,m in models.items()})
print('两侧样本:', evidence['payload']['by_version'])
print('真实KV账本:', evidence['payload']['kv'])
print('门禁:',verdict)
print('实际KV/full最大差异:', evidence['payload']['max_cache_full_diff'])

两侧样本: {'stable': {'n': 12, 'bad': 0, 'latency_exceeded': 0, 'errors': 0, 'latest_ordinal': 22}, 'canary': {'n': 12, 'bad': 0, 'latency_exceeded': 0, 'errors': 0, 'latest_ordinal': 23}}
真实KV账本: {'live_blocks': 0, 'leaked_sessions': 0, 'peak_live_blocks': 3, 'page_bytes': 512, 'capacity_blocks': 32}
门禁: {'action': 'allow', 'reason': 'fresh_healthy_bound_evidence', 'evidence_hash': '5f6f78d28ca52e72dce4385fc326510d02c71924db81f79267005966518bb9fb'}
实际KV/full最大差异: 2.384185791015625e-07


### 证据失效比“坏”更重要

一个干净的健康报告，如果来自**上一个 epoch**、几十个逻辑请求之前，或者对应已经改写的 candidate，就不能用于当前发布。

本课把两类情况区分开：

1. 摘要或身份不匹配 → 明确拒绝，不能把损坏当“稍后重试”。
2. 数据来源真实但已过时 → `wait`，必须重采样并重审。

只修改报告中的 `bad=0` 而不更新摘要会被检测；即使恶意重算摘要，模型/计划绑定仍能发现**跨计划**错配。但同一文件系统上独立重写全部证据不是本课能抵御的威胁，需要外部签发与可信身份。

In [10]:
old_decision=gate_decision(evidence,intent,policy,
    current_tick=healthy_gateway.logical_tick+policy.evidence_ttl_ticks+1,
    current_epoch=healthy_gateway.router.epoch,
    current_hashes={k:state_hash(m.state_dict()) for k,m in models.items()})
print('过期证据:',old_decision)
tampered=copy.deepcopy(evidence)
tampered['payload']['by_version']['canary']['bad']=4
try:
    gate_decision(tampered,intent,policy,current_tick=healthy_gateway.logical_tick,
      current_epoch=healthy_gateway.router.epoch,
      current_hashes={k:state_hash(m.state_dict()) for k,m in models.items()})
except LessonError as exc:
    print('内容篡改被拒绝:',exc)

过期证据: {'action': 'wait', 'reason': 'stale_evidence_or_epoch', 'evidence_hash': '5f6f78d28ca52e72dce4385fc326510d02c71924db81f79267005966518bb9fb'}
内容篡改被拒绝: 证据摘要不匹配


## 5. 人工审批需要绑定**同一证据版本**

教学审批有两个职责：`model-reviewer` 和 `oncall-operator`，且标识独立、不可由发布作者自批。审批绑定计划、门禁策略、证据摘要和逻辑 tick。门禁变更、证据过期或角色缺失，都不能继续发布。

**这两个身份是模拟角色，不是实际 IAM、电子签名或 MFA。** 程序内字符串不能证明人类真的批准；不能把这里的 `approval.sha256` 写成密码学不可抵赖签名。生产场景需要由外部可信身份系统签发批准事件。

[Google SRE：Release Engineering and Canarying](https://sre.google/workbook/canarying-releases/)讨论的是实际发布过程和可以回滚的变更，而非仅仅生成一个本地状态字段。

In [11]:
# %% 06 教学角色审批、原子发布指针与哈希链操作日志

def make_approval(intent:ReleaseIntent,evidence:dict,policy:GatePolicy,
                  *,principal:str,role:str,tick:int) -> dict:
    require(type(principal) is str and re.fullmatch('[A-Za-z0-9_-]{3,48}',principal),
            '模拟审批人标识非法')
    require(role in {'model-reviewer','oncall-operator'},'不允许的审批职责')
    require(principal!=intent.author,'发布作者不能自行审批')
    require(type(tick) is int and tick>=0,'审批tick非法')
    payload={'release_id':intent.release_id,'intent_hash':digest(asdict(intent)),
             'evidence_hash':evidence['sha256'],'policy_hash':digest(asdict(policy)),
             'principal':principal,'role':role,'tick':tick,
             'kind':'simulated_review_not_authenticated_signature'}
    return {'payload':payload,'sha256':digest(payload)}


def validate_approvals(approvals:list[dict],intent:ReleaseIntent,evidence:dict,
                       policy:GatePolicy,*,now_tick:int) -> dict:
    require(type(approvals) is list,'审批需要列表')
    observed_roles={};identities=set()
    for item in approvals:
        require(type(item) is dict and set(item)=={'payload','sha256'}
                and item['sha256']==digest(item['payload']),'审批摘要或形状不一致')
        p=item['payload']
        require(p['kind']=='simulated_review_not_authenticated_signature',
                '不能把教学审批冒充密码学签名')
        require(p['release_id']==intent.release_id and p['intent_hash']==digest(asdict(intent))
                and p['evidence_hash']==evidence['sha256'] and p['policy_hash']==digest(asdict(policy)),
                '审批绑定的发布计划或证据不一致')
        require(p['role'] in {'model-reviewer','oncall-operator'},'职责非法')
        require(p['principal']!=intent.author and p['principal'] not in identities,
                '作者自审或同一标识重复审批')
        require(p['role'] not in observed_roles,'同一职责重复审批')
        require(type(now_tick) is int and type(p['tick']) is int and
                p['tick']<=now_tick<=p['tick']+policy.approval_ttl_ticks,
                '审批过期或来自未来')
        observed_roles[p['role']]=p['principal'];identities.add(p['principal'])
    require(set(observed_roles)=={'model-reviewer','oncall-operator'},'缺少两种独立审批角色')
    return observed_roles


def atomic_json(path:Path,value:dict) -> None:
    """同一目录temp→fsync(file)→replace→fsync(dir)，仅本地POSIX教学语义。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    descriptor,tmp=tempfile.mkstemp(prefix='.next-',suffix='.json',dir=str(path.parent))
    try:
        with os.fdopen(descriptor,'w',encoding='utf-8') as handle:
            handle.write(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2,
                                    allow_nan=False)+'\n')
            handle.flush();os.fsync(handle.fileno())
        os.replace(tmp,path)
        if os.name=='posix':
            fd=os.open(str(path.parent),os.O_RDONLY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)


class HashJournal:
    """Append-only 的教学哈希链；没有外部可信根，不能抵御整体重写。"""
    def __init__(self,path):
        self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
        if not self.path.exists():self.path.touch()

    def verify(self):
        rows=[];prev='0'*64
        with self.path.open('r',encoding='utf-8') as stream:
            for n,line in enumerate(stream):
                require(bool(line.strip()),'journal包含空行/损坏')
                row=json.loads(line)
                require(set(row)=={'seq','event','op_id','payload','prev','sha256'},
                        '操作日志字段不符')
                require(row['seq']==n and row['prev']==prev,'日志序号或前向链接错误')
                require(row['sha256']==digest({k:v for k,v in row.items() if k!='sha256'}),
                        '日志记录摘要被改写')
                require(row['event'] in {'prepare','commit','abort','postcheck','runbook','incident'},
                        '操作日志事件类型非法')
                prev=row['sha256'];rows.append(row)
        prepared={};ended=set()
        for row in rows:
            op=row['op_id']
            if row['event']=='prepare':
                require(op not in prepared,'重复prepare操作')
                prepared[op]=row
            elif row['event'] in {'commit','abort'}:
                require(op in prepared and op not in ended,'没有prepare或重复终态')
                ended.add(op)
        return {'rows':rows,'root_hash':prev,'count':len(rows),
                'pending':{key:row for key,row in prepared.items() if key not in ended}}

    def append(self,event,op_id,payload):
        require(type(op_id) is str and re.fullmatch('[A-Za-z0-9_-]{3,64}',op_id),
                '操作ID非法')
        require(type(payload) is dict and len(canonical(payload))<8000,'日志细节非法')
        old=self.verify()
        row={'seq':old['count'],'event':event,'op_id':op_id,'payload':copy.deepcopy(payload),
             'prev':old['root_hash']}
        row['sha256']=digest(row)
        require(event in {'prepare','commit','abort','postcheck','runbook','incident'},
                '事件不允许')
        with self.path.open('a',encoding='utf-8') as stream:
            stream.write(canonical(row)+'\n');stream.flush();os.fsync(stream.fileno())
        self.verify()
        return row


class SimulatedCrash(RuntimeError):
    """可重复的故障点，不触发真正的进程/系统宕机。"""


class ReleaseStore:
    """一次一个协调者；没有跨进程锁或分布式一致性，不能冒充真实发布控制面。"""
    def __init__(self,folder:Path,intent:ReleaseIntent):
        intent.validate()
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.intent=intent
        self.pointer=self.folder/'active_pointer.json'
        self.journal=HashJournal(self.folder/'operations.jsonl')
        if not self.pointer.exists():
            require(self.journal.verify()['count']==0,'日志已存在而指针缺失，禁止初始化覆盖')
            atomic_json(self.pointer,{'generation':intent.start_generation,
                      'active':'stable','model_hash':intent.stable_hash,
                      'release_id':None,'previous':None})
        self.read()

    def read(self):
        result=json.loads(self.pointer.read_text(encoding='utf-8'))
        require(set(result)=={'generation','active','model_hash','release_id','previous'},
                '活动指针结构异常')
        require(type(result['generation']) is int and result['generation']>=0,'generation非法')
        require(result['active'] in {'stable','canary'},'活动版本非法')
        require(result['model_hash']==(self.intent.stable_hash if result['active']=='stable'
                                      else self.intent.candidate_hash),'活动指针与模型权重身份不符')
        if result['active']=='canary':
            require(result['release_id']==self.intent.release_id,'候选不属于本发布计划')
        return result

    def switch(self,to:str,op_id:str,*,expected_generation:int,crash_at=None,
               evidence_hash:str|None=None):
        require(to in {'stable','canary'},'发布目标非法')
        require(crash_at in {None,'after_prepare','after_pointer'},'故障点非法')
        journal=self.journal.verify();require(not journal['pending'],'有未恢复事务禁止下一次切换')
        current=self.read()
        require(current['generation']==expected_generation,'旧generation；拒绝覆盖新版本')
        require(current['active']!=to,'目标已经激活，不可重复发布')
        next_value={'generation':current['generation']+1,'active':to,
                    'model_hash':self.intent.candidate_hash if to=='canary' else self.intent.stable_hash,
                    'release_id':self.intent.release_id if to=='canary' else None,
                    'previous':current['active']}
        require(not any(r['op_id']==op_id for r in journal['rows']),'操作ID必须全局单次使用')
        plan={'before':current,'after':next_value,'intent_hash':digest(asdict(self.intent)),
              'evidence_hash':evidence_hash,'kind':'promote' if to=='canary' else 'rollback'}
        self.journal.append('prepare',op_id,plan)
        if crash_at=='after_prepare':raise SimulatedCrash('故意中断：prepare后、pointer前')
        atomic_json(self.pointer,next_value)
        if crash_at=='after_pointer':raise SimulatedCrash('故意中断：pointer后、commit记录前')
        self.journal.append('commit',op_id,{'pointer_sha256':digest(next_value)})
        return self.read()

    def recover(self):
        report=self.journal.verify();pending=report['pending'];pointer=self.read();actions=[]
        require(len(pending)<=1,'同时存在多个悬而未决的发布操作，需要人工处理')
        for op,prepared in pending.items():
            plan=prepared['payload']
            require(plan['intent_hash']==digest(asdict(self.intent)),'恢复时发布计划不匹配')
            if pointer==plan['before']:
                self.journal.append('abort',op,{'reason':'pointer_not_switched'})
                actions.append({'op_id':op,'status':'aborted_no_switch'})
            elif pointer==plan['after']:
                self.journal.append('commit',op,{'pointer_sha256':digest(pointer),
                                                 'recovered_after_pointer':True})
                actions.append({'op_id':op,'status':'committed_existing_pointer'})
            else:
                raise LessonError('日志pending与当前指针均不一致，需要人工对账，禁止猜测恢复')
        require(not self.journal.verify()['pending'],'恢复后仍有未决操作')
        return {'pointer':self.read(),'actions':actions,'root_hash':self.journal.verify()['root_hash']}

In [12]:
approval_review = make_approval(intent,evidence,policy,principal='reviewerA',
    role='model-reviewer',tick=healthy_gateway.logical_tick)
approval_ops = make_approval(intent,evidence,policy,principal='operatorB',
    role='oncall-operator',tick=healthy_gateway.logical_tick)
print('模拟双角色审批:', validate_approvals([approval_review,approval_ops],
    intent,evidence,policy,now_tick=healthy_gateway.logical_tick))
try:
    validate_approvals([approval_review],intent,evidence,policy,
                       now_tick=healthy_gateway.logical_tick)
except LessonError as exc:
    print('单角色不能发布:',exc)

模拟双角色审批: {'model-reviewer': 'reviewerA', 'oncall-operator': 'operatorB'}
单角色不能发布: 缺少两种独立审批角色


## 6. 发布指针：写日志、原子替换、崩溃后对账

本课本地状态更新顺序：

```text
读取当前active指针和generation
   ↓
prepare操作写入日志并flush/fsync
   ↓
同目录临时文件写入、fsync，再os.replace活动指针
   ↓
向日志写入commit并fsync
```

如果在 `prepare` 后、切指针前崩溃，指针仍旧，应将准备操作标记 `abort`。

如果已经切指针、但尚未写入 `commit` 就崩溃，读取指针等于计划后的内容，应补写 `commit`，**不能虚构未发布**。

日志与指针既不是两阶段共识协议，也不提供跨进程互斥、跨机器事务和供应链可信认证。本课明确限定为同一目录、本地 POSIX、单个协调者的教学模式。摘要链只能检测非恶意遗漏/单独改写，需要可信根才能进一步证明来源。

In [13]:
with tempfile.TemporaryDirectory(prefix='l80-journal-example-') as tmp:
    state=ReleaseStore(Path(tmp),intent)
    print('初始active:',state.read())
    try:
        state.switch('canary','example-crash',expected_generation=0,crash_at='after_pointer')
    except SimulatedCrash as exc:
        print('注入故障:',exc)
    print('实际指针（已切换）:',state.read())
    print('未决事务数:',len(state.journal.verify()['pending']))
    print('安全对账:',state.recover())
    print('重复对账不重放:',state.recover()['actions'])

初始active: {'active': 'stable', 'generation': 0, 'model_hash': '2d30d4ca305e81b68830c7c05211f92fa3d82c79dba1d31150174a56162c2c30', 'previous': None, 'release_id': None}
注入故障: 故意中断：pointer后、commit记录前
实际指针（已切换）: {'active': 'canary', 'generation': 1, 'model_hash': '2842e64e3e133a65eb122e843af66381dfd8828f91bec39392ea1cc38be0230b', 'previous': 'stable', 'release_id': 'rel-80-demo'}
未决事务数: 1
安全对账: {'pointer': {'active': 'canary', 'generation': 1, 'model_hash': '2842e64e3e133a65eb122e843af66381dfd8828f91bec39392ea1cc38be0230b', 'previous': 'stable', 'release_id': 'rel-80-demo'}, 'actions': [{'op_id': 'example-crash', 'status': 'committed_existing_pointer'}], 'root_hash': 'b6369cb12536e5bd73b2b69ae51cee861613823d12469747a38d163b4a740158'}
重复对账不重放: []


## 7. 发布控制器：Runbook要验证**实际动作**

同一份控制器按顺序要求：门禁允许、两角色审批有效、旧请求排空、KV归零、generation一致、持久日志无悬挂事务。完成后才切向候选，并对**新唯一请求**验证版本、SLO与KV释放。

回滚不仅是写 `rolled_back=True`：应从当前活动指针切回已知稳定权重，generation 单调递增，并在下一批新请求上重测。已有准入请求不会在中途偷偷换模型。

Runbook在每一步列出失败分支（阻止、等待、转人工、对账、回滚），而不仅是“发布命令”。

In [14]:
# %% 07 发布控制器、受控验证、Runbook 和无责事故复盘
class ReleaseController:
    def __init__(self,gateway:IncrementalGateway,intent:ReleaseIntent,policy:GatePolicy,
                 folder:Path):
        self.gateway=gateway;self.intent=intent;self.policy=policy
        self.store=ReleaseStore(folder,intent)
        self.approvals=[];self.evidence=None;self.last_decision=None
        self.state='draft'

    def model_hashes(self):
        return {n:state_hash(m.state_dict()) for n,m in self.gateway.models.items()}

    def attach(self,evidence):
        require(self.state in {'draft','reviewing'},'不能在发布/事故后替换证据')
        self.evidence=copy.deepcopy(evidence)
        self.last_decision=gate_decision(evidence,self.intent,self.policy,
                              current_tick=self.gateway.logical_tick,
                              current_epoch=self.gateway.router.epoch,
                              current_hashes=self.model_hashes())
        self.state='reviewing'
        return self.last_decision

    def approve(self,role,principal):
        require(self.state=='reviewing' and self.evidence is not None,'尚未进入审核状态')
        require(self.last_decision['action']=='allow','门禁不是allow，不得审批发布')
        candidate=make_approval(self.intent,self.evidence,self.policy,principal=principal,
                                role=role,tick=self.gateway.logical_tick)
        # 先检查，后写入；失败的审批不得污染状态。
        principals=[x['payload']['principal'] for x in self.approvals]
        roles=[x['payload']['role'] for x in self.approvals]
        require(principal not in principals and role not in roles,
                '审批职责或模拟身份重复')
        self.approvals.append(candidate)
        return candidate

    def promote(self,*,crash_at=None):
        require(self.state=='reviewing','当前状态不能发布')
        current=gate_decision(self.evidence,self.intent,self.policy,
                              current_tick=self.gateway.logical_tick,
                              current_epoch=self.gateway.router.epoch,
                              current_hashes=self.model_hashes())
        require(current['action']=='allow' and current==self.last_decision,
                '证据过期、模型变化或门禁不再允许发布')
        validate_approvals(self.approvals,self.intent,self.evidence,self.policy,
                           now_tick=self.gateway.logical_tick)
        require(self.gateway.pool.usage()['live_blocks']==0 and
                not any(not rec.completed for rec in self.gateway.registry.values()),
                '发布前必须排空会话/物理KV，不允许把运行中会话切换权重')
        before=self.store.read()
        require(before['active']=='stable','发布前的active必须是稳定版本')
        op_id='promote-'+self.intent.release_id
        try:
            result=self.store.switch('canary',op_id,
                       expected_generation=before['generation'],crash_at=crash_at,
                       evidence_hash=self.evidence['sha256'])
        except SimulatedCrash:
            self.state='recovering';raise
        self.gateway.router.set_weight(100)
        self.state='promoted'
        return result

    def recover(self):
        report=self.store.recover()
        self.gateway.router.set_weight(100 if report['pointer']['active']=='canary' else 0)
        self.state='promoted' if report['pointer']['active']=='canary' else 'reviewing'
        return report

    def rollback(self,*,reason='operator_requested'):
        require(self.state=='promoted','只有已经发布的候选才需要指针回滚')
        require(reason in {'operator_requested','postcheck_failed','incident_detected'},'回滚原因非法')
        require(self.store.read()['active']=='canary','没有候选活动指针')
        require(self.model_hashes()['stable']==self.intent.stable_hash,'已知良好稳定权重不匹配')
        require(not any(not rec.completed for rec in self.gateway.registry.values()),
                '回滚前请明确排空已有请求；不能偷偷改运行中请求版本')
        old=self.store.read()
        result=self.store.switch('stable','rollback-'+self.intent.release_id,
                                 expected_generation=old['generation'])
        self.gateway.router.set_weight(0)
        self.state='rolled_back'
        return result

    def verify_new_traffic(self,count=8):
        require(self.state in {'promoted','rolled_back'},'需要发布/回滚后才能验证新流量')
        require(type(count) is int and count>=6,'恢复样本量至少6')
        expected='canary' if self.state=='promoted' else 'stable'
        require(self.store.read()['active']==expected,'指针与状态不一致')
        start=len(self.gateway.terminal)
        results=[]
        for i in range(count):
            key=f'after_{expected}_{start}_{i}'
            row,_=self.gateway.serve(ServiceRequest('orgA',key,'hello',timeout_ticks=70),fault='none')
            results.append(row)
        require(all(row['version']==expected and row['slo_good'] for row in results),
                '发布/回滚后新流量并非全部健康的目标版本')
        require(len({x['trace_id'] for x in results})==count,'新流量不能包含重试')
        require(self.gateway.pool.usage()['live_blocks']==0 and not self.gateway.leaked_sessions,
                '验证后真实KV必须完全释放')
        self.store.journal.append('postcheck','verify-'+self.intent.release_id+'-'+expected,
                                  {'version':expected,'count':count,'slo_bad':0,
                                   'kv_live_blocks':0})
        return {'version':expected,'count':count,'good':count,
                'kv_live_blocks':self.gateway.pool.usage()['live_blocks'],
                'pointer_generation':self.store.read()['generation']}


def generate_runbook(intent:ReleaseIntent,policy:GatePolicy):
    return {'release_id':intent.release_id,'roles':{'review':'model-reviewer',
             'operations':'oncall-operator','author':intent.author},
            'stages':[
                {'step':'preflight','check':'固定base/adapter/tokenizer/template与真实增量KV一致性',
                 'on_failure':'block_and_investigate'},
                {'step':'canary','check':f'稳定版和候选各至少{policy.minimum_per_version}个唯一终态',
                 'on_failure':'wait_or_block'},
                {'step':'gate','check':'模型身份、证据/审批新鲜度、SLO、物理KV账本',
                 'on_failure':'wait_block_or_page'},
                {'step':'approve','check':'两个不同模拟角色，独立标识，不能由作者自审',
                 'on_failure':'no_promotion'},
                {'step':'commit','check':'prepare journal→atomic pointer→commit journal',
                 'on_failure':'recover_and_reconcile'},
                {'step':'verify','check':'新请求仅走当前指针版本，成功且KV释放',
                 'on_failure':'rollback_or_operator'},
                {'step':'review','check':'证据关联、未知项和具验证标准的行动项',
                 'on_failure':'keep_postmortem_open'},
             ],
            'boundaries':['time_is_logical_ticks','offline_one_controller_no_external_lock',
                          'simulated_approvals_are_not_authentication',
                          'hash_chain_not_a_trusted_signature','random_model_not_language_quality']}


def build_postmortem(name:str,evidence:dict,decision:dict,action:str,
                     *,controlled_injection:str,incident_observation=None) -> dict:
    require(name in {'healthy','candidate_slow','global_slow','kv_leak',
                     'crash_before_pointer','crash_after_pointer','postrelease_incident'},'未知演练')
    require(action in {'promoted','blocked','paged','rolled_back'},'未知控制动作')
    e=evidence['payload'];numbers=e['by_version']
    observations={v:{'terminal_count':numbers[v]['n'], 'bad':numbers[v]['bad'],
                     'latency_exceeded':numbers[v]['latency_exceeded'],
                     'errors':numbers[v]['errors']} for v in ('stable','canary')}
    observations['physical_kv']=copy.deepcopy(e['kv'])
    observations['max_cache_full_diff']=e['max_cache_full_diff']
    if incident_observation is not None:
        require(name=='postrelease_incident' and type(incident_observation) is dict
                and set(incident_observation)=={'count','bad'}
                and type(incident_observation['count']) is int
                and 0<=incident_observation['bad']<=incident_observation['count'],
                '发布后事故观测格式错误')
        observations['after_promotion']=copy.deepcopy(incident_observation)
    # 注入模式是测试夹具掌握的事实，不等于真实生产事故能直接推断的根因。
    hypotheses={'healthy':['当前样本未暴露可见回归；不保证生产永远健康'],
                'candidate_slow':['候选版路径可能包含额外开销，需检查候选特有trace'],
                'global_slow':['两版本共用资源或负载可能是共同影响因素'],
                'kv_leak':['候选推理的会话清理或引用计数路径可能未释放'],
                'crash_before_pointer':['prepare日志已写入，指针未切换；对账应撤销pending'],
                'crash_after_pointer':['指针已切换而commit未落盘；应补记事务终态'],
                'postrelease_incident':['发布后候选新增慢终态，先恢复稳定路由，再排查候选路径']}
    actions=[
        {'id':'act-monitor-'+name,'owner_role':'oncall-operator','priority':'P1',
         'status':'open','verification':'复演相同负载并核对唯一终态SLO及KV零占用'},
        {'id':'act-test-'+name,'owner_role':'model-reviewer','priority':'P2',
         'status':'open','verification':'为该失效模式添加自动失败测试并记录重放摘要'},
    ]
    return {'scenario':name,'release_id':e['release_id'],
            'classification':'synthetic_controlled_drill_not_real_incident',
            'controlled_injection':controlled_injection,
            'observed':observations,'gate':copy.deepcopy(decision),
            'action_taken':action,
            'timeline':[{'logical_tick':e['sample_tick'],'stage':'evidence_captured'},
                        {'logical_tick':e['sample_tick'],'stage':'gate_'+decision['action']},
                        {'logical_tick':e['sample_tick'],'stage':action}],
            'hypotheses_not_proven_root_causes':hypotheses[name],
            'unknowns':['尚无真实HTTP/GPU或跨节点部署信号',
                        '没有外部可信身份签名或生产观测分母'],
            'followups':actions,'review_status':'draft_requires_human_review',
            'privacy':'no_prompts_tenants_idempotency_keys_or_raw_text'}


def postmortem_markdown(report):
    return '\n'.join([
        '# 教学事故复盘：'+report['scenario'],
        '', '**状态：** 草稿，等待人工审阅；不是生产事故报告。',
        '', '## 已观测事实',
        '', '```json',json.dumps(report['observed'],ensure_ascii=False,indent=2,sort_keys=True),'```',
        '', '## 门禁与实际动作',
        '', f"门禁 `{report['gate']['action']}`：`{report['gate']['reason']}`；动作 `{report['action_taken']}`。",
        '', '## 待验证假设（不是已确认根因）',
        *['- '+p for p in report['hypotheses_not_proven_root_causes']],
        '', '## 未知与数据边界',
        *['- '+p for p in report['unknowns']],
        '', '## 行动项（仍是开放状态）',
        *['- '+a['id']+' / '+a['owner_role']+' / '+a['verification'] for a in report['followups']],
        '', '没有把模拟审批、哈希摘要或逻辑tick冒充真实签名与生产时延。',
        '' ])

In [15]:
with tempfile.TemporaryDirectory(prefix='l80-controller-example-') as tmp:
    manager=ReleaseController(healthy_gateway,intent,policy,Path(tmp))
    manager.attach(evidence)
    manager.approve('model-reviewer','reviewerA')
    manager.approve('oncall-operator','operatorB')
    print('发布后:',manager.promote())
    print('新候选请求验证:',manager.verify_new_traffic(count=8))
    print('实际回滚:',manager.rollback(reason='operator_requested'))
    print('新稳定请求验证:',manager.verify_new_traffic(count=8))
    print('操作日志:', manager.store.journal.verify()['count'], '条')

发布后: {'active': 'canary', 'generation': 1, 'model_hash': '2842e64e3e133a65eb122e843af66381dfd8828f91bec39392ea1cc38be0230b', 'previous': 'stable', 'release_id': 'rel-80-demo'}
新候选请求验证: {'version': 'canary', 'count': 8, 'good': 8, 'kv_live_blocks': 0, 'pointer_generation': 1}
实际回滚: {'active': 'stable', 'generation': 2, 'model_hash': '2d30d4ca305e81b68830c7c05211f92fa3d82c79dba1d31150174a56162c2c30', 'previous': 'canary', 'release_id': None}
新稳定请求验证: {'version': 'stable', 'count': 8, 'good': 8, 'kv_live_blocks': 0, 'pointer_generation': 2}
操作日志: 6 条


## 8. 七组事故与故障演练：不是每种问题都应“自动回滚”

| 演练 | 阶段 | 正确动作 |
|---|---|---|
| 健康候选 | 发布门禁 | 双职责审核后发布并验证 |
| 仅候选变慢 | 发布门禁 | `block`，指针本来就没动，**不能冒称已回滚** |
| 两个版本都慢 | 发布门禁 | `page`，共享退化不能简单归咎候选 |
| 候选 KV 泄漏 | 发布门禁 | `block`，清理实际物理页 |
| prepare 后中断 | 本地发布操作 | 旧指针未改→安全标记abort |
| 指针切换后中断 | 本地发布操作 | 旧日志未commit→对账并补记、再有据回滚 |
| 发布之后候选变慢 | 发布后观测 | 执行新一代回滚，再验证稳定流量 |

场景是可重复的离线教学故障，不是生产真实事故；推理的 K/V 确实用物理 Tensor 计算。

In [16]:
# %% 08 五组真实增量KV发布演练与独立文件回放

def simulate_release_case(name:str,tok,models,policy:GatePolicy,folder:Path):
    require(name in {'healthy','candidate_slow','global_slow','kv_leak',
                     'crash_before_pointer','crash_after_pointer','postrelease_incident'},
            '未知发布演练')
    fault=({'healthy':'none','candidate_slow':'slow','global_slow':'global_slow',
            'kv_leak':'kv_leak','crash_before_pointer':'none',
            'crash_after_pointer':'none','postrelease_incident':'none'})[name]
    gateway=IncrementalGateway(tok,models,DrillConfig())
    intent=make_intent(tok,models,release_id='rel80-'+name.replace('_','-'))
    gateway.router.start_canary(50)
    used=set()
    traffic=stage_traffic(gateway,{'stable':policy.minimum_per_version,
                                   'canary':policy.minimum_per_version},used,mode=fault)
    require(len(traffic)==2*policy.minimum_per_version,'演练真实终态不足')
    evidence=capture_release_evidence(gateway,intent,policy)
    manager=ReleaseController(gateway,intent,policy,folder)
    decision=manager.attach(evidence)
    extra={'scenario':name,'fault_injected':fault,'initial_gate':decision,
           'unique_executions':gateway.execution_count,'stage_unique_terminals':len(traffic),
           'stage_gateway_metrics':gateway.metrics(),
           'evidence':evidence,'runbook':generate_runbook(intent,policy)}
    if name=='kv_leak':
        require(decision['action']=='block','未发现KV泄漏')
        extra['manual_release_leaks']=gateway.release_leaks()
        extra['action']='blocked'
    elif name=='candidate_slow':
        require(decision['action']=='block','未阻止候选慢请求')
        extra['action']='blocked'
    elif name=='global_slow':
        require(decision['action']=='page','全局慢请求应人工处置')
        extra['action']='paged'
    else:
        require(decision['action']=='allow','健康候选不应被拒绝')
        manager.approve('model-reviewer','reviewerA')
        manager.approve('oncall-operator','operatorB')
        extra['simulated_approvals']=copy.deepcopy(manager.approvals)
        if name=='crash_before_pointer':
            try: manager.promote(crash_at='after_prepare')
            except SimulatedCrash as err:extra['crash_message']=str(err)
            else:raise AssertionError('prepare之后未触发故障')
            extra['pending_after_crash']=len(manager.store.journal.verify()['pending'])
            extra['recovery']=manager.recover()
            require(manager.store.read()['active']=='stable','未改变指针不能声称已发布')
            extra['action']='aborted_by_recovery'
        elif name=='crash_after_pointer':
            try:manager.promote(crash_at='after_pointer')
            except SimulatedCrash as err:extra['crash_message']=str(err)
            else:raise AssertionError('pointer之后未触发故障')
            extra['pending_after_crash']=len(manager.store.journal.verify()['pending'])
            extra['recovery']=manager.recover()
            require(manager.store.read()['active']=='canary','指针已经切换，恢复不得谎称未发布')
            extra['post_promote']=manager.verify_new_traffic()
            manager.rollback(reason='operator_requested')
            extra['post_rollback']=manager.verify_new_traffic()
            extra['action']='reconciled_and_rolled_back'
        else:
            extra['promoted_pointer']=manager.promote()
            extra['post_promote']=manager.verify_new_traffic()
            if name=='postrelease_incident':
                # 事故发生在发布之后，先取得真实的慢终态，再执行“指针回滚”。
                known=len(gateway.terminal)
                bad=[]
                for i in range(8):
                    row,_=gateway.serve(ServiceRequest('orgA',f'incident_{known}_{i}','hello'),fault='slow')
                    bad.append(row)
                require(all(r['version']=='canary' and not r['slo_good'] for r in bad),
                        '故障注入应只影响活动候选版本')
                extra['postrelease_bad']={'count':len(bad),'bad':sum(not r['slo_good'] for r in bad)}
                manager.rollback(reason='incident_detected')
                extra['post_rollback']=manager.verify_new_traffic()
                extra['action']='rolled_back'
            else:
                extra['action']='promoted'
    # 被拒绝的候选只能停止新灰度准入；指针本来就没切换，不冒称已做生产回滚。
    if manager.store.read()['active']=='stable':gateway.router.set_weight(0)
    extra['final_pointer']=manager.store.read()
    extra['operations']=manager.store.journal.verify()['rows']
    extra['journal_root_hash']=manager.store.journal.verify()['root_hash']
    extra['physical_kv_end']=gateway.pool.usage()['live_blocks']
    extra['pending_operations']=len(manager.store.journal.verify()['pending'])
    extra['independent_model_hashes']={k:state_hash(m.state_dict()) for k,m in models.items()}
    extra['postmortem']=build_postmortem(
        name,evidence,decision,'promoted' if extra['action']=='promoted' else
        'rolled_back' if extra['action'] in {'reconciled_and_rolled_back','rolled_back'} else
        'paged' if extra['action']=='paged' else 'blocked',controlled_injection=fault,
        incident_observation=extra.get('postrelease_bad'))
    require(extra['physical_kv_end']==0,'演练结束时仍有活跃物理KV页')
    require(extra['pending_operations']==0,'演练结束仍有未完成发布事务')
    return extra


def run_experiment(output_root=Path('lesson_80_outputs'),seed=8001):
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    out=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    out.mkdir(parents=True,exist_ok=False)
    tok,models=build_servable_models(seed)
    policy=GatePolicy();policy.validate()
    intent=make_intent(tok,models)
    settings={'version':VERSION,'seed':seed,'torch':str(torch.__version__),
              'python':platform.python_version(),'policy':asdict(policy),
              'intent_demo':asdict(intent),'sources':SOURCES,
              'limitations':['offline CPU real incremental KV tensors','synthetic arrival order / logical ticks',
                             'fixed two fake approval identities: not authentication',
                             'one writer / no cross-process lock or real orchestrator',
                             'atomic rename and fsync are not distributed transactions',
                             'blameless reports are drafts; no real user data or inferred root cause']}
    write_json(out/'settings.json',settings)
    save_models(out,tok,models)
    names=['healthy','candidate_slow','global_slow','kv_leak',
           'crash_before_pointer','crash_after_pointer','postrelease_incident']
    reports={}
    for name in names:
        reports[name]=simulate_release_case(name,tok,models,policy,out/'cases'/name)
        (out/'postmortems').mkdir(exist_ok=True)
        (out/'postmortems'/f'{name}.md').write_text(
             postmortem_markdown(reports[name]['postmortem']),encoding='utf-8')
    write_json(out/'scenario_reports.json',reports)
    summary={name:{'gate':r['initial_gate']['action'],'gate_reason':r['initial_gate']['reason'],
                   'action':r['action'],'pointer_active':r['final_pointer']['active'],
                   'generation':r['final_pointer']['generation'],
                   'kv_live_blocks':r['physical_kv_end'],
                   'pending':r['pending_operations']} for name,r in reports.items()}
    write_json(out/'summary.json',summary)
    write_json(out/'runbook.json',generate_runbook(intent,policy))
    reload_report=reload_run(out)
    write_json(out/'reload.json',reload_report)
    for name,row in summary.items():print(name,canonical(row))
    print('输出目录:',out.resolve())
    return out


def reload_run(folder):
    folder=Path(folder)
    settings=json.loads((folder/'settings.json').read_text(encoding='utf-8'))
    require(settings['version']==VERSION,'版本不匹配')
    tok,models=load_models(folder)
    policy=GatePolicy(**settings['policy']);policy.validate()
    saved=json.loads((folder/'scenario_reports.json').read_text(encoding='utf-8'))
    require(set(saved)=={'healthy','candidate_slow','global_slow','kv_leak',
                         'crash_before_pointer','crash_after_pointer','postrelease_incident'},
            '回放场景集合不完整')
    rereports={}
    # 新独立子目录，避免覆盖已有发布指针；模型是从检查点重新建立的对象。
    with tempfile.TemporaryDirectory(prefix='lesson80-replay-') as temp:
        for name in saved:
            rereports[name]=simulate_release_case(name,tok,models,policy,Path(temp)/name)
    require(rereports==saved,'独立重载的决策、指针、日志或事故复盘不同')
    # 另外验证已发布指针的哈希账本。只读，不修改原文件。
    for name,r in saved.items():
        store=ReleaseStore(folder/'cases'/name,
              make_intent(tok,models,release_id='rel80-'+name.replace('_','-')))
        require(store.read()==r['final_pointer'],'磁盘active指针与报告不同')
        trace=store.journal.verify()
        require(trace['root_hash']==r['journal_root_hash'] and not trace['pending'],
                '发布日志链与报告不一致')
    return {'replay_equal':True,'cases':len(saved),'all_physical_kv_released':True,
            'all_journals_closed':True,'scope':'independent CPU replay + read-only release state verify'}

In [17]:
experiment_path=None
if RUN_FULL_EXPERIMENT:
    experiment_path=run_experiment(OUTPUT_ROOT,seed=8001)
    summary=json.loads((experiment_path/'summary.json').read_text(encoding='utf-8'))
    print('门禁/活动版本/Generation/动作：')
    for name,row in summary.items():
        print(name,row['gate'],row['pointer_active'],row['generation'],row['action'])
else:
    print('本轮只学习机制：未执行完整事故演练，也没有制造成绩。')

healthy {"action":"promoted","gate":"allow","gate_reason":"fresh_healthy_bound_evidence","generation":1,"kv_live_blocks":0,"pending":0,"pointer_active":"canary"}
candidate_slow {"action":"blocked","gate":"block","gate_reason":"candidate_slo_regression","generation":0,"kv_live_blocks":0,"pending":0,"pointer_active":"stable"}
global_slow {"action":"paged","gate":"page","gate_reason":"control_also_unhealthy","generation":0,"kv_live_blocks":0,"pending":0,"pointer_active":"stable"}
kv_leak {"action":"blocked","gate":"block","gate_reason":"unreleased_physical_kv","generation":0,"kv_live_blocks":0,"pending":0,"pointer_active":"stable"}
crash_before_pointer {"action":"aborted_by_recovery","gate":"allow","gate_reason":"fresh_healthy_bound_evidence","generation":0,"kv_live_blocks":0,"pending":0,"pointer_active":"stable"}
crash_after_pointer {"action":"reconciled_and_rolled_back","gate":"allow","gate_reason":"fresh_healthy_bound_evidence","generation":2,"kv_live_blocks":0,"pending":0,"pointer_act

## 9. 故障复盘：事实、操作、根因假设不能混写

演练报告分开记录：

1. **可追溯事实**：每侧唯一终态与实际坏数、逻辑耗时、KV物理占用、已执行发布动作。
2. **判断与控制动作**：门禁的明确理由、当前指针和 generation、人工升级还是回滚。
3. **假设与未知项**：不能从“候选变慢”直接写出“已确定开发者写错了某一行”。
4. **改进项**：责任主体为职责角色，包含可以复验的验收标准，初始状态为开放，不能伪称已解决。

本课输出的是`draft_requires_human_review`，不是完成审核的真实事故文档。Google SRE 的无责复盘强调客观分析与后续改进行动，而不是将事故归罪于个体。[SRE postmortem culture](https://sre.google/sre-book/postmortem-culture/)

In [18]:
if experiment_path is not None:
    r=json.loads((experiment_path/'scenario_reports.json').read_text(encoding='utf-8'))
    for name in ['candidate_slow','global_slow','kv_leak']:
        report=r[name]['postmortem']
        print(name,'门禁:',report['gate']['action'],'实际动作:',report['action_taken'])
        print('已观测:',report['observed'])
        print('假设（未确认）:',report['hypotheses_not_proven_root_causes'])
else:
    print('尚未执行完整演练，没有复盘报告。')

candidate_slow 门禁: block 实际动作: blocked
已观测: {'canary': {'bad': 12, 'errors': 0, 'latency_exceeded': 12, 'terminal_count': 12}, 'max_cache_full_diff': 2.384185791015625e-07, 'physical_kv': {'capacity_blocks': 32, 'leaked_sessions': 0, 'live_blocks': 0, 'page_bytes': 512, 'peak_live_blocks': 3}, 'stable': {'bad': 0, 'errors': 0, 'latency_exceeded': 0, 'terminal_count': 12}}
假设（未确认）: ['候选版路径可能包含额外开销，需检查候选特有trace']
global_slow 门禁: page 实际动作: paged
已观测: {'canary': {'bad': 12, 'errors': 0, 'latency_exceeded': 12, 'terminal_count': 12}, 'max_cache_full_diff': 2.384185791015625e-07, 'physical_kv': {'capacity_blocks': 32, 'leaked_sessions': 0, 'live_blocks': 0, 'page_bytes': 512, 'peak_live_blocks': 3}, 'stable': {'bad': 12, 'errors': 0, 'latency_exceeded': 12, 'terminal_count': 12}}
假设（未确认）: ['两版本共用资源或负载可能是共同影响因素']
kv_leak 门禁: block 实际动作: blocked
已观测: {'canary': {'bad': 2, 'errors': 2, 'latency_exceeded': 0, 'terminal_count': 12}, 'max_cache_full_diff': 2.384185791015625e-07, 'physical_kv': {'

## 10. 文件、恢复和“不可抵赖”边界

完整运行目录主要包含：

| 文件 | 内容 |
|---|---|
| `base.pt`、`stable.pt`、`canary.pt` | 确定的模型/adapter内容与身份绑定 |
| `scenario_reports.json` | 七组证据、审批、状态、事故与资源报告 |
| `summary.json`、`runbook.json` | 汇总与可执行阶段清单 |
| `cases/*/active_pointer.json` | 每个独立演练的持久活动版本指针 |
| `cases/*/operations.jsonl` | 本地append、逐条hash的操作日志 |
| `postmortems/*.md` | 无责复盘草稿，等待真正人工审阅 |
| `reload.json` | 独立新对象的事故回放与只读校验 |

教学日志可检测单条记录篡改，但没有链根外部锚点或真实身份签名，不能宣称抵御拥有整个目录写权限的攻击者。真实 CodeDeploy 的回滚是一次独立的**新部署**，不是魔法式擦除历史。 [AWS Rollback 文档](https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html)

In [19]:
if experiment_path is not None:
    print('独立新模型回放 + 只读状态核对：')
    print(json.dumps(reload_run(experiment_path),ensure_ascii=False,indent=2))
else:
    print('没有运行目录，跳过本次磁盘重载。')

独立新模型回放 + 只读状态核对：


{
  "replay_equal": true,
  "cases": 7,
  "all_physical_kv_released": true,
  "all_journals_closed": true,
  "scope": "independent CPU replay + read-only release state verify"
}


## 11. 固定检查：失败要真正拒绝、恢复要有证据

此处跑完整回归，不只看七个情景是否打印了结果。检查真实KV logits、双侧分母、旧epoch/词表/模型失效、审批过期与自审、审计篡改、prepare崩溃对账、悬挂会话、回滚新流量与复盘草稿。

所有检查通过不代表真实 IAM、数据库共识、云发布安全或生产评测完成。

In [20]:
# %% 09 固定回归检查：本体、证据、审批、发布事务和事故复盘
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,result):
        if not bool(result):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,a,b):self.ok(name,a==b)
    def close(self,name,a,b,tol=1e-7):self.ok(name,abs(float(a)-float(b))<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+'：错误输入应被拒绝')


def run_checks():
    torch.set_num_threads(1)
    c=Checks();seed=8080;tok,models=build_servable_models(seed);policy=GatePolicy()
    intent=make_intent(tok,models,release_id='rel80-check')
    c.equal('固定词表大小',tok.vocab_size,281)
    c.equal('UTF8编码往返',tok.decode(tok.encode('hello 42')),'hello 42')
    c.ok('对话模板BOS存在',tok.apply_chat_template([{'role':'user','content':'hello'}],add_generation_prompt=True)[0]==1)
    c.ok('两个版本权重不同',intent.stable_hash!=intent.candidate_hash)
    c.equal('底座权重相同',base_hash(models['stable']),base_hash(models['canary']))
    c.ok('两个模型全部冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('独立权重存储',all(a.data_ptr()!=b.data_ptr() for a,b in zip(models['stable'].parameters(),models['canary'].parameters())))
    c.equal('tokenizer摘要绑定',intent.tokenizer_hash,digest(tok.contract()))
    c.rejects('候选与stable同一权重禁止',lambda:ReleaseIntent('relX',intent.stable_hash,intent.stable_hash,intent.tokenizer_hash,'chat-v1','author').validate())
    c.rejects('发布作者非法标识拒绝',lambda:ReleaseIntent('relX',intent.stable_hash,intent.candidate_hash,intent.tokenizer_hash,'chat-v1','!').validate())
    c.rejects('证据TTL为0拒绝',lambda:GatePolicy(evidence_ttl_ticks=0).validate())
    c.rejects('KV容差无效拒绝',lambda:GatePolicy(cache_tolerance=float('nan')).validate())
    c.rejects('最小样本不足拒绝',lambda:GatePolicy(minimum_per_version=5).validate())

    # KV逐步真实计算，对稳定/候选分别与完整前向核对。
    kvpool=PagedKVPool(32,4,models['stable'].cfg,{'stable','canary'})
    runner=CachedModelRunner(tok,models,kvpool)
    prompt=tok.apply_chat_template([{'role':'user','content':'hello'}],tokenize=True,add_generation_prompt=True)
    for version in ('stable','canary'):
        sid='check_'+version
        runner.open_prefill(sid,version,prompt)
        c.ok('真实KV预填充与full一致_'+version,runner.assert_consistent(sid)<2e-5)
        next_token=int(runner.last_logits[sid][16:].argmax())+16
        runner.step(sid,next_token)
        c.ok('真实KV增量解码与full一致_'+version,runner.assert_consistent(sid)<2e-5)
        c.equal('KV会话锁定模型版本_'+version,kvpool.sessions[sid].version,version)
        runner.close(sid)
    c.equal('预检KV完整释放',kvpool.usage()['live_blocks'],0)
    c.equal('KV无活跃会话',kvpool.usage()['session_count'],0)
    c.ok('真实张量占用内存',kvpool.data.numel()>0)
    c.ok('真实KV计算token数记录',runner.tokens_processed>0)

    def fixture(mode='none',count=12):
        g=IncrementalGateway(tok,models,DrillConfig())
        g.router.start_canary(50)
        rows=stage_traffic(g,{'stable':count,'canary':count},set(),mode)
        e=capture_release_evidence(g,intent,policy)
        return g,e,rows
    gateway,evidence,rows=fixture('none')
    payload=evidence['payload']
    c.equal('证据两侧各12个终态',sorted(x['n'] for x in payload['by_version'].values()),[12,12])
    c.equal('真实请求执行数',gateway.execution_count,24)
    c.equal('唯一分母',payload['unique_trace_count'],24)
    c.ok('真实模型logits_KV差异有限',payload['max_cache_full_diff']<2e-5)
    c.equal('健康场景物理KV释放',payload['kv']['live_blocks'],0)
    c.equal('stage epoch正确',payload['sample_epoch'],1)
    c.equal('证据使用实际逻辑tick',payload['sample_tick'],gateway.logical_tick)
    c.ok('没有原始请求正文在证据',not any(k in canonical(payload) for k in ('"text"','"tenant"','"key"','"prompt"')))
    def decide(e=evidence,g=gateway,i=intent,p=policy,**kwargs):
        return gate_decision(e,i,p,current_tick=kwargs.get('tick',g.logical_tick),
              current_epoch=kwargs.get('epoch',g.router.epoch),
              current_hashes=kwargs.get('hashes',{n:state_hash(m.state_dict()) for n,m in g.models.items()}))
    decision=decide()
    c.equal('健康门禁allow',decision['action'],'allow')
    c.equal('决策绑定证据摘要',decision['evidence_hash'],evidence['sha256'])
    c.equal('完全相同输入决定性重算',decide(),decision)
    c.equal('证据到期等待',decide(tick=gateway.logical_tick+policy.evidence_ttl_ticks+1)['action'],'wait')
    c.equal('恰好TTL边界仍可使用',decide(tick=gateway.logical_tick+policy.evidence_ttl_ticks)['action'],'allow')
    c.equal('灰度epoch改变证据失效',decide(epoch=2)['action'],'wait')
    bad_e=copy.deepcopy(evidence);bad_e['payload']['by_version']['canary']['bad']=5
    c.rejects('单独改证据数值被摘要发现',lambda:decide(e=bad_e))
    bad_e=copy.deepcopy(evidence);bad_e['payload']['tokenizer_hash']='0'*64
    bad_e['sha256']=digest(bad_e['payload'])
    c.rejects('整体重算摘要仍不能绕过词表绑定',lambda:decide(e=bad_e))
    bad_i=copy.deepcopy(intent)
    bad_i=ReleaseIntent(**{**asdict(intent),'release_id':'rel80-other'})
    c.rejects('证据不属于别的发布意图',lambda:decide(i=bad_i))
    c.rejects('现存模型身份改变拒绝',lambda:decide(hashes={'stable':'0'*64,'canary':intent.candidate_hash}))
    c.rejects('当前逻辑时间不能倒退',lambda:decide(tick=payload['sample_tick']-1))
    small=copy.deepcopy(evidence)
    small['payload']['by_version']['canary']['n']=4
    small['payload']['by_version']['stable']['n']=20
    small['sha256']=digest(small['payload'])
    c.equal('单侧样本不足等待',decide(e=small)['action'],'wait')
    half=copy.deepcopy(evidence)
    half['payload']['last_ordinal']=100
    half['sha256']=digest(half['payload'])
    c.equal('旧版本遥远终态不可放行',decide(e=half)['action'],'wait')
    memory=copy.deepcopy(evidence);memory['payload']['kv']['live_blocks']=1
    memory['sha256']=digest(memory['payload'])
    c.equal('物理KV残留阻止发布',decide(e=memory)['reason'],'unreleased_physical_kv')
    oom=copy.deepcopy(evidence);oom['payload']['kv']['peak_live_blocks']=33
    oom['payload']['kv']['capacity_blocks']=64;oom['sha256']=digest(oom['payload'])
    c.equal('峰值超过预算被阻止',decide(e=oom)['reason'],'peak_kv_exceeds_budget')
    logit=copy.deepcopy(evidence);logit['payload']['max_cache_full_diff']=.1
    logit['sha256']=digest(logit['payload'])
    c.equal('KV与full偏差过大被阻止',decide(e=logit)['reason'],'cache_vs_full_mismatch')
    dup=copy.deepcopy(evidence);dup['payload']['unique_trace_count']=20
    dup['sha256']=digest(dup['payload'])
    c.rejects('重试不能增加独立终态分母',lambda:decide(e=dup))
    slow_g,slow_e,_=fixture('slow')
    c.equal('仅候选慢请求阻止发布',decide(e=slow_e,g=slow_g)['reason'],'candidate_slo_regression')
    global_g,global_e,_=fixture('global_slow')
    c.equal('双版本全差转人工',decide(e=global_e,g=global_g)['action'],'page')
    leak_g,leak_e,_=fixture('kv_leak')
    c.equal('真实KV泄漏被阻止',decide(e=leak_e,g=leak_g)['reason'],'unreleased_physical_kv')
    c.ok('泄漏场景确有实际物理占用',leak_g.pool.usage()['live_blocks']>0)
    c.ok('泄漏清理实际释放会话',leak_g.release_leaks()>0)
    c.equal('人工清理后KV无遗留',leak_g.pool.usage()['live_blocks'],0)
    # 模拟审批与双角色；摘要自洽不等于外部真实身份认证。
    a=make_approval(intent,evidence,policy,principal='reviewerA',role='model-reviewer',tick=gateway.logical_tick)
    b=make_approval(intent,evidence,policy,principal='operatorB',role='oncall-operator',tick=gateway.logical_tick)
    c.equal('两角色审批通过',len(validate_approvals([a,b],intent,evidence,policy,now_tick=gateway.logical_tick)),2)
    c.rejects('缺第二角色拒绝',lambda:validate_approvals([a],intent,evidence,policy,now_tick=gateway.logical_tick))
    c.rejects('同一人跨职责自批拒绝',lambda:validate_approvals([a,make_approval(intent,evidence,policy,
                     principal='reviewerA',role='oncall-operator',tick=gateway.logical_tick)],intent,evidence,policy,now_tick=gateway.logical_tick))
    c.rejects('作者自审拒绝',lambda:make_approval(intent,evidence,policy,
                  principal=intent.author,role='model-reviewer',tick=gateway.logical_tick))
    c.rejects('未知模拟角色拒绝',lambda:make_approval(intent,evidence,policy,
                  principal='other',role='security-admin',tick=gateway.logical_tick))
    c.rejects('审批超时不可复用',lambda:validate_approvals([a,b],intent,evidence,policy,
                            now_tick=gateway.logical_tick+policy.approval_ttl_ticks+1))
    c.equal('恰好审批过期边界仍允许',len(validate_approvals([a,b],intent,evidence,policy,
                          now_tick=gateway.logical_tick+policy.approval_ttl_ticks)),2)
    modified=copy.deepcopy(a);modified['payload']['role']='oncall-operator'
    c.rejects('审批信息篡改检测',lambda:validate_approvals([modified,b],intent,evidence,policy,
                       now_tick=gateway.logical_tick))
    c.rejects('审批跨证据复用拒绝',lambda:validate_approvals([a,b],intent,slow_e,policy,
                       now_tick=gateway.logical_tick))
    different=GatePolicy(maximum_bad_fraction=.15)
    c.rejects('审批门禁策略变化须重审',lambda:validate_approvals([a,b],intent,evidence,different,
                       now_tick=gateway.logical_tick))
    # 不污染原发布演练gateway；新控制器重建一个独立的健康网关。
    with tempfile.TemporaryDirectory() as tmp:
        root=Path(tmp)
        manager=ReleaseController(gateway,intent,policy,root/'happy')
        c.equal('新发布指针初始stable',manager.store.read()['active'],'stable')
        c.equal('新发布指针generation0',manager.store.read()['generation'],0)
        c.equal('绑定健康证据成功',manager.attach(evidence)['action'],'allow')
        c.rejects('缺少两角色不得promote',lambda:manager.promote())
        c.equal('审批缺失后仍stable',manager.store.read()['active'],'stable')
        manager.approve('model-reviewer','reviewerA')
        c.rejects('一人审批不得发布',lambda:manager.promote())
        c.rejects('重复角色审批拒绝',lambda:manager.approve('model-reviewer','reviewerC'))
        c.rejects('角色伪造拒绝',lambda:manager.approve('arbitrary','unknown'))
        c.equal('失败审批不能污染审批列表',len(manager.approvals),1)
        manager.approve('oncall-operator','operatorB')
        # 已准入但尚未执行的请求在切指针前应被拒绝；新请求不允许中途改版本。
        dangling,_=gateway.admit(ServiceRequest('orgA','pending_case','hello'))
        c.rejects('未排空请求不能提交发布',lambda:manager.promote())
        gateway.execute(dangling)
        old=manager.store.read()
        active=manager.promote()
        c.equal('promote生成单调递增',active['generation'],old['generation']+1)
        c.equal('promote活动版本candidate',active['active'],'canary')
        c.equal('发布路由切向候选',gateway.router.weight,100)
        c.rejects('重复promote不允许',lambda:manager.promote())
        c.equal('发布后真正新流量全走canary',manager.verify_new_traffic()['version'],'canary')
        c.equal('真实KV释放',gateway.pool.usage()['live_blocks'],0)
        c.rejects('旧generation不允许覆盖指针',lambda:manager.store.switch('stable','wrong-gen',
                         expected_generation=0))
        c.rejects('同目标重复切换拒绝',lambda:manager.store.switch('canary','repeat',
                         expected_generation=1))
        new=manager.rollback(reason='incident_detected')
        c.equal('回滚形成新generation',new['generation'],2)
        c.equal('活动指针恢复stable',new['active'],'stable')
        c.equal('回滚后候选准入权重0',gateway.router.weight,0)
        c.equal('回滚后新稳定流量验证',manager.verify_new_traffic()['version'],'stable')
        c.equal('回滚后KV无残留',gateway.pool.usage()['live_blocks'],0)
        c.rejects('不允许重复回滚',lambda:manager.rollback())
        c.ok('正常发布日志闭环',not manager.store.journal.verify()['pending'])
        c.ok('日志确有prepare与commit',all(event in [x['event'] for x in manager.store.journal.verify()['rows']]
                                        for event in ('prepare','commit','postcheck')))
        forged=manager.store.journal.path
        raw=forged.read_text(encoding='utf-8')
        forged.write_text(raw.replace('"prepare"','"tamper"',1),encoding='utf-8')
        c.rejects('单条操作日志篡改可检测',lambda:manager.store.journal.verify())
        forged.write_text(raw,encoding='utf-8')
        # 第二种篡改保持合法事件名，仅修改细节；必须依赖摘要而不是白名单检测。
        require('pointer_sha256' in raw,'准备的journal缺少commit摘要字段')
        forged.write_text(raw.replace('pointer_sha256','pointer_sha25x',1),encoding='utf-8')
        c.rejects('合法事件的操作细节变动须由哈希检查发现',lambda:manager.store.journal.verify())
        forged.write_text(raw,encoding='utf-8')
        c.ok('恢复原始日志可验证',manager.store.journal.verify()['count']>0)

    # 单独执行崩溃前后两个安全点的事务对账。
    for crash_point,status,revision in [('after_prepare','aborted_no_switch','stable'),
                                        ('after_pointer','committed_existing_pointer','canary')]:
        with tempfile.TemporaryDirectory() as tmp:
            store=ReleaseStore(Path(tmp),intent)
            try:store.switch('canary','case-'+crash_point,
                             expected_generation=0,crash_at=crash_point)
            except SimulatedCrash:pass
            else:raise AssertionError('没有触发教学故障')
            c.equal('故障点pending为1_'+crash_point,len(store.journal.verify()['pending']),1)
            result=store.recover()
            c.equal('恢复动作_'+crash_point,result['actions'][0]['status'],status)
            c.equal('故障后活动指针_'+crash_point,result['pointer']['active'],revision)
            c.equal('恢复后pending清零_'+crash_point,len(store.journal.verify()['pending']),0)
            c.equal('重复恢复无副作用_'+crash_point,len(store.recover()['actions']),0)
            c.rejects('重复操作ID拒绝_'+crash_point,lambda:store.switch(
                     'stable' if revision=='canary' else 'canary','case-'+crash_point,
                     expected_generation=result['pointer']['generation']))
    # 证据出处与复盘不把故障假设写成已验证根因。
    runbook=generate_runbook(intent,policy)
    c.equal('Runbook包含7个有序阶段',len(runbook['stages']),7)
    c.ok('Runbook每阶段有失败路径',all(x.get('on_failure') for x in runbook['stages']))
    c.ok('Runbook包含事故处理负责人角色','operations' in runbook['roles'])
    post=build_postmortem('candidate_slow',slow_e,
            decide(e=slow_e,g=slow_g),'blocked',controlled_injection='slow')
    c.equal('复盘明确synthetic边界',post['classification'],'synthetic_controlled_drill_not_real_incident')
    c.equal('复盘是草稿而非已审定根因',post['review_status'],'draft_requires_human_review')
    c.ok('复盘观察事实含两侧独立分母',all(v in post['observed'] for v in ('stable','canary')))
    c.ok('复盘根因只是假设',len(post['hypotheses_not_proven_root_causes'])>0)
    c.ok('事故行动项有验证标准',all(x['verification'] and x['owner_role'] for x in post['followups']))
    c.ok('事故行动项并未伪称已完成',all(x['status']=='open' for x in post['followups']))
    c.ok('Markdown包含未知与证据边界','## 未知与数据边界' in postmortem_markdown(post))
    c.ok('复盘不记录明文prompt',not any(k in canonical(post) for k in ('"text"','"tenant"','"key"')))
    # 主流程的七种情况也应遵守不同的决策，而不是将page当rollback。
    with tempfile.TemporaryDirectory() as tmp:
        desired={
           'healthy':('allow','promoted','canary',1),
           'candidate_slow':('block','blocked','stable',0),
           'global_slow':('page','paged','stable',0),
           'kv_leak':('block','blocked','stable',0),
           'crash_before_pointer':('allow','aborted_by_recovery','stable',0),
           'crash_after_pointer':('allow','reconciled_and_rolled_back','stable',2),
           'postrelease_incident':('allow','rolled_back','stable',2),
        }
        for name,(gate,action,active,generation) in desired.items():
            r=simulate_release_case(name,tok,models,policy,Path(tmp)/name)
            c.equal('七场景门禁_'+name,r['initial_gate']['action'],gate)
            c.equal('七场景动作_'+name,r['action'],action)
            c.equal('七场景活动指针_'+name,r['final_pointer']['active'],active)
            c.equal('七场景代际_'+name,r['final_pointer']['generation'],generation)
            c.equal('七场景无物理页泄漏_'+name,r['physical_kv_end'],0)
            c.equal('七场景无未决事务_'+name,r['pending_operations'],0)
    return {'passed':len(c.names),'names':c.names,
            'scope':'离线CPU真实KV+模拟审批/发布；不代表生产安全认证'}

In [21]:
checks=run_checks()
print('通过的回归检查数:',checks['passed'])
if experiment_path is not None:
    write_json(experiment_path/'checks.json',checks)

通过的回归检查数: 158


## 12. 独立练习 A：双版本发布门禁的优先级

实现 `gate_ex(stable_n, stable_bad, candidate_n, candidate_bad, min_n, max_bad)`，返回 `allow/wait/block/page`。

在固定分母足够后，**先检查稳定版本**：如果稳定本身坏率超阈值，返回 `page` 而不是仅仅拒绝候选；若只有候选超限则 `block`。输入坏数不能超过总数。建议先手算 `12条中坏3条` 的状态。

In [22]:
def check_exercise_a(fn):
    assert fn(12,0,12,0,12,.1)=='allow'
    assert fn(11,0,12,0,12,.1)=='wait'
    assert fn(12,3,12,0,12,.1)=='page'
    assert fn(12,0,12,3,12,.1)=='block'
    assert fn(12,1,12,1,12,.1)=='allow'
    try:fn(12,-1,12,0,12,.1)
    except (ValueError,LessonError):pass
    else:raise AssertionError('练习A需要拒绝坏分母')
    print('练习A：两侧独立样本与门禁优先级通过')


def check_exercise_b(fn):
    tok,models=build_servable_models(8050);policy=GatePolicy();intent=make_intent(tok,models)
    g=IncrementalGateway(tok,models,DrillConfig());g.router.start_canary(50)
    stage_traffic(g,{'stable':12,'canary':12},set())
    ev=capture_release_evidence(g,intent,policy)
    a=make_approval(intent,ev,policy,principal='reviewerA',role='model-reviewer',tick=g.logical_tick)
    b=make_approval(intent,ev,policy,principal='operatorB',role='oncall-operator',tick=g.logical_tick)
    assert fn([a,b],intent,ev,policy,g.logical_tick) is True
    for wrong in ([a],[a,a]):
        try:fn(wrong,intent,ev,policy,g.logical_tick)
        except (ValueError,LessonError):pass
        else:raise AssertionError('练习B必须要求独立角色')
    try:fn([a,b],intent,ev,policy,g.logical_tick+policy.approval_ttl_ticks+1)
    except (ValueError,LessonError):pass
    else:raise AssertionError('练习B不能使用过期审批')
    print('练习B：模拟双角色审批与证据绑定通过')


def check_exercise_c(fn):
    before={'generation':0,'active':'stable'}
    after={'generation':1,'active':'canary'}
    assert fn(before,after,before)=='abort'
    assert fn(before,after,after)=='commit'
    for wrong in ({'generation':2,'active':'canary'},
                  {'generation':1,'active':'stable'}):
        try:fn(before,after,wrong)
        except (ValueError,LessonError):pass
        else:raise AssertionError('练习C必须拒绝未知指针状态')
    print('练习C：prepare之后的指针对账通过')

In [23]:
def gate_ex(stable_n, stable_bad, candidate_n, candidate_bad, min_n, max_bad):
    raise NotImplementedError('请独立实现双版本门禁')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    check_exercise_a(gate_ex)

## 13. 独立练习 B：两角色模拟审批

实现 `approve_ex(approvals, intent, evidence, policy, now_tick)`，返回 `True`，或在违例时抛异常。

检查摘要、发布意图/门禁/证据绑定；需要 `model-reviewer` 与 `oncall-operator` 两种职责，不能是同一个标识，不能由作者自批。审批在有效期末 tick 仍有效，再晚一 tick 就过期。

记得：检查字符串和哈希**仍不等于身份认证**。

In [24]:
def approve_ex(approvals, intent, evidence, policy, now_tick):
    raise NotImplementedError('请独立实现模拟双职责审批校验')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    check_exercise_b(approve_ex)

## 14. 独立练习 C：恢复时对账，不猜测“是否切换成功”

实现 `reconcile_ex(before, after, pointer)`：如果活动指针还是旧计划，返回 `'abort'`；等于新计划返回 `'commit'`；既不等于旧也不等于新的状态必须抛错，不允许将未知指针归入成功。

核对 `after.generation == before.generation+1`；重复执行恢复时不应再次切指针。

In [25]:
def reconcile_ex(before, after, pointer):
    raise NotImplementedError('请独立实现prepare悬挂时的指针对账')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    check_exercise_c(reconcile_ex)

## 15. 完整参考答案：与练习实现不同名

先自行完成再运行。参考答案作为单独函数，不覆盖练习区的三个 `*_ex` 入口。

In [26]:
def answer_a(stable_n,stable_bad,canary_n,canary_bad,min_n,max_bad):
    require(all(type(x) is int for x in (stable_n,stable_bad,canary_n,canary_bad,min_n))
            and min_n>0 and stable_n>=0 and canary_n>=0
            and 0<=stable_bad<=stable_n and 0<=canary_bad<=canary_n,'输入分母/坏数非法')
    require(0<max_bad<1,'坏率阈值非法')
    if stable_n<min_n or canary_n<min_n:return 'wait'
    if stable_bad/stable_n>max_bad:return 'page'
    if canary_bad/canary_n>max_bad:return 'block'
    return 'allow'


def answer_b(approvals,intent,evidence,policy,now_tick):
    required={'model-reviewer','oncall-operator'}
    require(type(approvals) is list,'需要审批列表')
    roles=set();principals=set()
    for a in approvals:
        require(a['sha256']==digest(a['payload']),'审批摘要不符')
        p=a['payload']
        require(p['release_id']==intent.release_id and p['intent_hash']==digest(asdict(intent))
                and p['policy_hash']==digest(asdict(policy))
                and p['evidence_hash']==evidence['sha256'],'审批关联不符')
        require(p['role'] in required and p['role'] not in roles,'角色缺失/重复')
        require(p['principal'] not in principals and p['principal']!=intent.author,'职责不能自批')
        require(p['tick']<=now_tick<=p['tick']+policy.approval_ttl_ticks,'审批过期')
        roles.add(p['role']);principals.add(p['principal'])
    require(roles==required,'两个不同职责必须齐全')
    return True


def answer_c(before,after,pointer):
    require(before!=after and before['generation']+1==after['generation'],
            'prepare代际须前进1')
    if pointer==before:return 'abort'
    if pointer==after:return 'commit'
    raise LessonError('当前指针不是旧/新计划任何一个版本，必须人工干预')

In [27]:
check_exercise_a(answer_a)
check_exercise_b(answer_b)
check_exercise_c(answer_c)

练习A：两侧独立样本与门禁优先级通过
练习B：模拟双角色审批与证据绑定通过
练习C：prepare之后的指针对账通过


## 16. 验收与下一课

本课毕业验收应能解释：

- `allow` 需要新鲜的双侧证据，不是“脚本返回0”。
- `block` 是**防止发布**，与发布后的 `rollback` 是两个动作。
- `page` 在稳定版也退化时比盲目切回旧版本更合适。
- 模拟审批不等于身份认证；哈希链没有外部锚点无法证明来源。
- prepare日志存在时，活动指针才是恢复的重要事实，不能只读最后一行日志猜结果。
- 复盘应分开事实、假设、行动项，并明确仍需人审。

**下一课第 81 课建议：推理服务灾难恢复、检查点与跨实例一致性。** 将单机指针事务扩展为故障切换、持久快照、恢复点/恢复时间目标与多副本验证，但不把它误称为无需共识的分布式原子发布。

## 资料

- [Google SRE Workbook：Canarying Releases](https://sre.google/workbook/canarying-releases/)
- [Google SRE Book：Postmortem Culture](https://sre.google/sre-book/postmortem-culture/)
- [Google SRE：Incident Management Guide](https://sre.google/resources/practices-and-processes/incident-management-guide/)
- [AWS CodeDeploy：Rollback and Redeploy](https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html)
- [PyTorch：Randomness and Reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)

**引用用于概念边界，本课数值来自本地实际执行，不是引用文献中的生产成绩。**

In [28]:
# %% 11 CLI：默认真实演练、检查、数学示例与独立重放

def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    group=parser.add_mutually_exclusive_group()
    group.add_argument('--checks-only',action='store_true')
    group.add_argument('--math-only',action='store_true')
    group.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_80_outputs'))
    parser.add_argument('--seed',type=int,default=8001)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2));return 0
    if args.math_only:
        print(json.dumps({'gate_example':answer_a(12,0,12,2,12,.1),
                          'release_state':{'prepare':'fsynced journal',
                                           'switch':'atomic local pointer',
                                           'recover':'reconcile pointer with pending operation'},
                          'limitations':'offline simulated logical ticks'},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定回归检查通过：{checks['passed']}项。真实CPU增量KV，模拟审批与本地指针。")
    if args.checks_only:return 0
    folder=run_experiment(args.output_dir,args.seed)
    write_json(folder/'checks.json',checks)
    return 0